In [ ]:
# Автоматическое создание структуры проекта Quoridor
import os

with open('go.mod', 'w', encoding='utf-8') as f:
    f.write('module quoridor\n\ngo 1.27.2\n')

os.makedirs('board', exist_ok=True)
with open('board/moves.go', 'w', encoding='utf-8') as f:
    f.write('package board\n\n// Move кодирует ход в uint16: старший байт — вид, младший — индекс.\n//\n//\tKindPawn: idx = клетка назначения 0..80\n//\tKindH/KindV: idx = r*8+c, как индекс бита в HW/VW\ntype Move uint16\n\nconst (\n\tKindPawn = 0\n\tKindH    = 1\n\tKindV    = 2\n)\n\nfunc PawnMove(to int) Move { return Move(to) }\n\nfunc WallMove(horizontal bool, r, c int) Move {\n\tk := KindV\n\tif horizontal {\n\t\tk = KindH\n\t}\n\treturn Move(k<<8 | (r*8 + c))\n}\n\nfunc (m Move) Kind() int { return int(m >> 8) }\nfunc (m Move) Idx() int  { return int(m & 0xff) }\n\n// Winner: 0 или 1, если соответствующая пешка на целевой строке, иначе -1.\nfunc (b *Board) Winner() int {\n\tfor p := 0; p < 2; p++ {\n\t\tif int(b.P[p])/9 == GoalRow(p) {\n\t\t\treturn p\n\t\t}\n\t}\n\treturn -1\n}\n\n// Apply применяет ЛЕГАЛЬНЫЙ ход и передаёт ход сопернику.\n// Легальность не проверяется (в горячем цикле поиска она уже известна из генератора).\nfunc (b Board) Apply(m Move) Board {\n\tswitch m.Kind() {\n\tcase KindPawn:\n\t\tb.P[b.Turn] = uint8(m.Idx())\n\tcase KindH:\n\t\tb.HW |= 1 << uint(m.Idx())\n\t\tb.Walls[b.Turn]--\n\tdefault:\n\t\tb.VW |= 1 << uint(m.Idx())\n\t\tb.Walls[b.Turn]--\n\t}\n\tb.Turn ^= 1\n\treturn b\n}\n\n// Moves добавляет в out все легальные ходы игрока, чей ход.\n// Стены проверяются медленной эталонной WithWall (полный BFS).\n// В терминальной позиции ходов нет.\nfunc (b *Board) Moves(out []Move) []Move {\n\tif b.Winner() >= 0 {\n\t\treturn out\n\t}\n\tvar buf [8]uint8\n\tfor _, to := range b.PawnMoves(int(b.Turn), buf[:0]) {\n\t\tout = append(out, PawnMove(int(to)))\n\t}\n\tif b.Walls[b.Turn] == 0 {\n\t\treturn out\n\t}\n\tfor r := 0; r < 8; r++ {\n\t\tfor c := 0; c < 8; c++ {\n\t\t\tif _, ok := b.WithWall(true, r, c); ok {\n\t\t\t\tout = append(out, WallMove(true, r, c))\n\t\t\t}\n\t\t\tif _, ok := b.WithWall(false, r, c); ok {\n\t\t\t\tout = append(out, WallMove(false, r, c))\n\t\t\t}\n\t\t}\n\t}\n\treturn out\n}\n\n// Perft считает число листьев на глубине depth (терминальные позиции листьев не дают).\nfunc Perft(b Board, depth int) uint64 {\n\tif depth == 0 {\n\t\treturn 1\n\t}\n\tms := b.Moves(make([]Move, 0, 160))\n\tif depth == 1 {\n\t\treturn uint64(len(ms))\n\t}\n\tvar n uint64\n\tfor _, m := range ms {\n\t\tn += Perft(b.Apply(m), depth-1)\n\t}\n\treturn n\n}\n\n')

os.makedirs('board', exist_ok=True)
with open('board/ref_test.go', 'w', encoding='utf-8') as f:
    f.write('package board\n\n// Независимая медленная эталонная реализация: другая модель данных\n// (список стен + проверка рёбер геометрически + DFS), ничего не берёт из board.go\n// кроме кодирования Move.\n\nimport (\n\t"math/rand"\n\t"sort"\n\t"testing"\n)\n\ntype refWall struct {\n\th    bool\n\tr, c int\n}\n\ntype refPos struct {\n\tpawn  [2][2]int // [игрок]{row,col}\n\twalls [2]int\n\tturn  int\n\tws    []refWall\n}\n\nfunc refNew() refPos {\n\treturn refPos{pawn: [2][2]int{{0, 4}, {8, 4}}, walls: [2]int{10, 10}}\n}\n\nfunc in9(r, c int) bool { return r >= 0 && r < 9 && c >= 0 && c < 9 }\n\n// blocked: заблокировано ли ребро между соседними клетками (a,b).\nfunc (p *refPos) blocked(r1, c1, r2, c2 int) bool {\n\tfor _, w := range p.ws {\n\t\tif r1 != r2 { // вертикальное перемещение, мешает горизонтальная стена\n\t\t\tlo := r1\n\t\t\tif r2 < lo {\n\t\t\t\tlo = r2\n\t\t\t}\n\t\t\tif w.h && w.r == lo && (c1 == w.c || c1 == w.c+1) {\n\t\t\t\treturn true\n\t\t\t}\n\t\t} else {\n\t\t\tlo := c1\n\t\t\tif c2 < lo {\n\t\t\t\tlo = c2\n\t\t\t}\n\t\t\tif !w.h && w.c == lo && (r1 == w.r || r1 == w.r+1) {\n\t\t\t\treturn true\n\t\t\t}\n\t\t}\n\t}\n\treturn false\n}\n\nfunc (p *refPos) reach(player int) bool {\n\tvar seen [9][9]bool\n\tvar dfs func(r, c int) bool\n\tdfs = func(r, c int) bool {\n\t\tif seen[r][c] {\n\t\t\treturn false\n\t\t}\n\t\tseen[r][c] = true\n\t\tgoal := 8\n\t\tif player == 1 {\n\t\t\tgoal = 0\n\t\t}\n\t\tif r == goal {\n\t\t\treturn true\n\t\t}\n\t\tfor _, d := range [4][2]int{{1, 0}, {-1, 0}, {0, 1}, {0, -1}} {\n\t\t\tnr, nc := r+d[0], c+d[1]\n\t\t\tif in9(nr, nc) && !p.blocked(r, c, nr, nc) && dfs(nr, nc) {\n\t\t\t\treturn true\n\t\t\t}\n\t\t}\n\t\treturn false\n\t}\n\treturn dfs(p.pawn[player][0], p.pawn[player][1])\n}\n\nfunc (p *refPos) wallLegal(w refWall) bool {\n\tif p.walls[p.turn] == 0 || w.r < 0 || w.r > 7 || w.c < 0 || w.c > 7 {\n\t\treturn false\n\t}\n\tfor _, o := range p.ws {\n\t\tif o.h == w.h {\n\t\t\tif w.h && o.r == w.r && abs(o.c-w.c) <= 1 {\n\t\t\t\treturn false\n\t\t\t}\n\t\t\tif !w.h && o.c == w.c && abs(o.r-w.r) <= 1 {\n\t\t\t\treturn false\n\t\t\t}\n\t\t} else if o.r == w.r && o.c == w.c {\n\t\t\treturn false\n\t\t}\n\t}\n\tq := *p\n\tq.ws = append(append([]refWall{}, p.ws...), w)\n\treturn q.reach(0) && q.reach(1)\n}\n\nfunc abs(x int) int {\n\tif x < 0 {\n\t\treturn -x\n\t}\n\treturn x\n}\n\nfunc (p *refPos) won() bool {\n\treturn p.pawn[0][0] == 8 || p.pawn[1][0] == 0\n}\n\nfunc (p *refPos) moves() []Move {\n\tvar out []Move\n\tif p.won() {\n\t\treturn out\n\t}\n\tme, op := p.pawn[p.turn], p.pawn[1-p.turn]\n\tdirs := [4][2]int{{1, 0}, {-1, 0}, {0, 1}, {0, -1}}\n\tfor _, d := range dirs {\n\t\tnr, nc := me[0]+d[0], me[1]+d[1]\n\t\tif !in9(nr, nc) || p.blocked(me[0], me[1], nr, nc) {\n\t\t\tcontinue\n\t\t}\n\t\tif nr != op[0] || nc != op[1] {\n\t\t\tout = append(out, PawnMove(nr*9+nc))\n\t\t\tcontinue\n\t\t}\n\t\tjr, jc := nr+d[0], nc+d[1]\n\t\tif in9(jr, jc) && !p.blocked(nr, nc, jr, jc) {\n\t\t\tout = append(out, PawnMove(jr*9+jc))\n\t\t\tcontinue\n\t\t}\n\t\tfor _, s := range dirs {\n\t\t\tif s[0]*d[0]+s[1]*d[1] != 0 { // только перпендикулярные\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\ttr, tc := nr+s[0], nc+s[1]\n\t\t\tif in9(tr, tc) && !p.blocked(nr, nc, tr, tc) {\n\t\t\t\tout = append(out, PawnMove(tr*9+tc))\n\t\t\t}\n\t\t}\n\t}\n\tfor r := 0; r < 8; r++ {\n\t\tfor c := 0; c < 8; c++ {\n\t\t\tfor _, h := range []bool{true, false} {\n\t\t\t\tif p.wallLegal(refWall{h, r, c}) {\n\t\t\t\t\tout = append(out, WallMove(h, r, c))\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t}\n\treturn out\n}\n\nfunc (p refPos) apply(m Move) refPos {\n\tp.ws = append([]refWall{}, p.ws...)\n\tswitch m.Kind() {\n\tcase KindPawn:\n\t\tp.pawn[p.turn] = [2]int{m.Idx() / 9, m.Idx() % 9}\n\tdefault:\n\t\tp.ws = append(p.ws, refWall{m.Kind() == KindH, m.Idx() / 8, m.Idx() % 8})\n\t\tp.walls[p.turn]--\n\t}\n\tp.turn = 1 - p.turn\n\treturn p\n}\n\nfunc refPerft(p refPos, d int) uint64 {\n\tif d == 0 {\n\t\treturn 1\n\t}\n\tms := p.moves()\n\tif d == 1 {\n\t\treturn uint64(len(ms))\n\t}\n\tvar n uint64\n\tfor _, m := range ms {\n\t\tn += refPerft(p.apply(m), d-1)\n\t}\n\treturn n\n}\n\nfunc sorted(ms []Move) []Move {\n\ts := append([]Move{}, ms...)\n\tsort.Slice(s, func(i, j int) bool { return s[i] < s[j] })\n\treturn s\n}\n\nfunc equalMoves(a, b []Move) bool {\n\ta, b = sorted(a), sorted(b)\n\tif len(a) != len(b) {\n\t\treturn false\n\t}\n\tfor i := range a {\n\t\tif a[i] != b[i] {\n\t\t\treturn false\n\t\t}\n\t}\n\treturn true\n}\n\nfunc TestPerftStart(t *testing.T) {\n\tb := New()\n\tif got := Perft(b, 1); got != 131 { // 3 хода пешкой + 128 стен\n\t\tt.Fatalf("perft1 = %d, want 131", got)\n\t}\n}\n\nfunc TestPerftVsReference(t *testing.T) {\n\tdepth := 3\n\tif testing.Short() {\n\t\tdepth = 2\n\t}\n\tfor d := 1; d <= depth; d++ {\n\t\tgot, want := Perft(New(), d), refPerft(refNew(), d)\n\t\tt.Logf("perft(%d) = %d (ref %d)", d, got, want)\n\t\tif got != want {\n\t\t\tt.Fatalf("perft(%d): board=%d ref=%d", d, got, want)\n\t\t}\n\t}\n}\n\n// Случайные партии: на каждой позиции наборы ходов должны совпасть.\nfunc TestRandomGamesVsReference(t *testing.T) {\n\trng := rand.New(rand.NewSource(1))\n\tgames, plies := 300, 120\n\tif testing.Short() {\n\t\tgames = 40\n\t}\n\tfor g := 0; g < games; g++ {\n\t\tb, p := New(), refNew()\n\t\tfor ply := 0; ply < plies; ply++ {\n\t\t\tbm := b.Moves(nil)\n\t\t\trm := p.moves()\n\t\t\tif !equalMoves(bm, rm) {\n\t\t\t\tt.Fatalf("game %d ply %d: move sets differ\\nboard=%v\\nref=%v\\nB=%+v", g, ply, sorted(bm), sorted(rm), b)\n\t\t\t}\n\t\t\tif (b.Winner() >= 0) != p.won() {\n\t\t\t\tt.Fatalf("winner mismatch game %d ply %d", g, ply)\n\t\t\t}\n\t\t\tif len(bm) == 0 {\n\t\t\t\tbreak\n\t\t\t}\n\t\t\t// чаще ходим пешкой, чтобы партии доходили до конца\n\t\t\tvar m Move\n\t\t\tif rng.Intn(100) < 55 {\n\t\t\t\tvar pm []Move\n\t\t\t\tfor _, x := range bm {\n\t\t\t\t\tif x.Kind() == KindPawn {\n\t\t\t\t\t\tpm = append(pm, x)\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\tif len(pm) > 0 {\n\t\t\t\t\tm = pm[rng.Intn(len(pm))]\n\t\t\t\t} else {\n\t\t\t\t\tm = bm[rng.Intn(len(bm))]\n\t\t\t\t}\n\t\t\t} else {\n\t\t\t\tm = bm[rng.Intn(len(bm))]\n\t\t\t}\n\t\t\tb, p = b.Apply(m), p.apply(m)\n\t\t\tif int(b.Turn) != p.turn || int(b.P[0]) != p.pawn[0][0]*9+p.pawn[0][1] ||\n\t\t\t\tint(b.P[1]) != p.pawn[1][0]*9+p.pawn[1][1] || b.Walls != [2]uint8{uint8(p.walls[0]), uint8(p.walls[1])} {\n\t\t\t\tt.Fatalf("state diverged game %d ply %d", g, ply)\n\t\t\t}\n\t\t}\n\t}\n}\n\n')

os.makedirs('board', exist_ok=True)
with open('board/extra_test.go', 'w', encoding='utf-8') as f:
    f.write('package board\n\nimport "testing"\n\nfunc has2(m []uint8, v int) bool {\n\tfor _, x := range m {\n\t\tif int(x) == v {\n\t\t\treturn true\n\t\t}\n\t}\n\treturn false\n}\n\nfunc TestEdgeJump(t *testing.T) { // соперник у края, прыжок прямо невозможен -> боковые\n\tb := New()\n\tb.P = [2]uint8{7*9 + 4, 8*9 + 4}\n\tm := b.PawnMoves(0, make([]uint8, 0, 8))\n\tif has2(m, 9*9+4) || !has2(m, 8*9+3) || !has2(m, 8*9+5) {\n\t\tt.Fatalf("edge: %v", m)\n\t}\n}\nfunc TestWallBetweenNoJump(t *testing.T) {\n\tb := New()\n\tb.P = [2]uint8{4*9 + 4, 5*9 + 4}\n\tb.HW |= 1 << uint(4*8+4) // стена между нами\n\tm := b.PawnMoves(0, make([]uint8, 0, 8))\n\tif has2(m, 5*9+4) || has2(m, 6*9+4) {\n\t\tt.Fatalf("%v", m)\n\t}\n}\nfunc TestSideBlockedToo(t *testing.T) {\n\tb := New()\n\tb.P = [2]uint8{4*9 + 4, 5*9 + 4}\n\tb.HW |= 1 << uint(5*8+4)\n\tb.VW |= 1 << uint(5*8+3) // слева от соперника\n\tm := b.PawnMoves(0, make([]uint8, 0, 8))\n\tif has2(m, 5*9+3) || !has2(m, 5*9+5) {\n\t\tt.Fatalf("%v", m)\n\t}\n}\nfunc TestPlayer1Moves(t *testing.T) {\n\tb := New()\n\tif m := b.PawnMoves(1, make([]uint8, 0, 8)); len(m) != 3 {\n\t\tt.Fatalf("%v", m)\n\t}\n}\nfunc TestSealIllegal(t *testing.T) {\n\tb := New()\n\tb.P[0] = 0\n\tb.HW |= 1 << 0\n\tif _, ok := b.WithWall(false, 0, 1); ok {\n\t\tt.Fatal("sealing wall must be illegal")\n\t}\n}\nfunc TestNoWalls(t *testing.T) {\n\tb := New()\n\tb.Walls[0] = 0\n\tif _, ok := b.WithWall(true, 3, 3); ok {\n\t\tt.Fatal("no walls left")\n\t}\n}\nfunc TestTurnFlip(t *testing.T) {\n\tb := New()\n\tb2, _ := b.WithWall(true, 3, 3)\n\tt.Logf("Turn after wall = %d (before %d)", b2.Turn, b.Turn)\n}\nfunc TestOverlapOtherSide(t *testing.T) {\n\tb := New()\n\tb2, _ := b.WithWall(true, 3, 3)\n\tif _, ok := b2.WithWall(true, 3, 2); ok {\n\t\tt.Fatal("left overlap")\n\t}\n\tif _, ok := b2.WithWall(true, 3, 5); !ok {\n\t\tt.Fatal("adjacent must be ok")\n\t}\n\tb3, _ := b.WithWall(false, 3, 3)\n\tif _, ok := b3.WithWall(false, 4, 3); ok {\n\t\tt.Fatal("vert overlap")\n\t}\n\tif _, ok := b3.WithWall(false, 2, 3); ok {\n\t\tt.Fatal("vert overlap up")\n\t}\n}\n\n')

os.makedirs('board', exist_ok=True)
with open('board/board.go', 'w', encoding='utf-8') as f:
    f.write('// Package board: базовое состояние Quoridor, правила и расстояния.\n//\n// Клетки: idx = row*9 + col, row/col в 0..8.\n// Игрок 0 стартует в (0,4) и идёт к строке 8; игрок 1 стартует в (8,4) и идёт к строке 0.\n//\n// Стены: индекс бита = r*8 + c, r,c в 0..7.\n//   HW (горизонтальная): лежит между строками r и r+1, закрывает столбцы c и c+1.\n//   VW (вертикальная):   лежит между столбцами c и c+1, закрывает строки r и r+1.\npackage board\n\nconst (\n\tUp = iota\n\tDown\n\tLeft\n\tRight\n)\n\nvar dr = [4]int{-1, 1, 0, 0}\nvar dc = [4]int{0, 0, -1, 1}\n\n// Для прыжка через пешку по направлению k боковые направления:\nvar sideDirs = [4][2]int{{Left, Right}, {Left, Right}, {Up, Down}, {Up, Down}}\n\ntype Board struct {\n\tHW, VW uint64\n\tP      [2]uint8 // позиции пешек\n\tWalls  [2]uint8 // стен в запасе\n\tTurn   uint8\n}\n\nfunc New() Board {\n\treturn Board{P: [2]uint8{4, 76}, Walls: [2]uint8{10, 10}}\n}\n\nfunc GoalRow(p int) int {\n\tif p == 0 {\n\t\treturn 8\n\t}\n\treturn 0\n}\n\nfunc has(m uint64, r, c int) bool {\n\tif r < 0 || r > 7 || c < 0 || c > 7 {\n\t\treturn false\n\t}\n\treturn m&(1<<uint(r*8+c)) != 0\n}\n\n// CanStep: можно ли шагнуть из (r,c) в направлении d (границы и стены).\nfunc (b *Board) CanStep(r, c, d int) bool {\n\tswitch d {\n\tcase Up:\n\t\treturn r > 0 && !has(b.HW, r-1, c) && !has(b.HW, r-1, c-1)\n\tcase Down:\n\t\treturn r < 8 && !has(b.HW, r, c) && !has(b.HW, r, c-1)\n\tcase Left:\n\t\treturn c > 0 && !has(b.VW, r, c-1) && !has(b.VW, r-1, c-1)\n\tdefault: // Right\n\t\treturn c < 8 && !has(b.VW, r, c) && !has(b.VW, r-1, c)\n\t}\n}\n\n// Dist: поле расстояний до строки goalRow без учёта пешек. -1 = недостижимо.\n// Рёбра симметричны, поэтому достаточно BFS от целевой строки. Без аллокаций.\nfunc (b *Board) Dist(goalRow int) [81]int8 {\n\tvar d [81]int8\n\tfor i := range d {\n\t\td[i] = -1\n\t}\n\tvar q [81]uint8\n\th, t := 0, 0\n\tfor c := 0; c < 9; c++ {\n\t\ti := goalRow*9 + c\n\t\td[i] = 0\n\t\tq[t] = uint8(i)\n\t\tt++\n\t}\n\tfor h < t {\n\t\tcur := int(q[h])\n\t\th++\n\t\tr, c := cur/9, cur%9\n\t\tfor k := 0; k < 4; k++ {\n\t\t\tif !b.CanStep(r, c, k) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tn := (r+dr[k])*9 + c + dc[k]\n\t\t\tif d[n] < 0 {\n\t\t\t\td[n] = d[cur] + 1\n\t\t\t\tq[t] = uint8(n)\n\t\t\t\tt++\n\t\t\t}\n\t\t}\n\t}\n\treturn d\n}\n\n// PawnMoves добавляет в out клетки, куда может пойти пешка p (с прыжками).\n// Передавай срез с предвыделенной ёмкостью (cap >= 5), чтобы не было аллокаций.\nfunc (b *Board) PawnMoves(p int, out []uint8) []uint8 {\n\tme, op := int(b.P[p]), int(b.P[1-p])\n\tr, c := me/9, me%9\n\tfor k := 0; k < 4; k++ {\n\t\tif !b.CanStep(r, c, k) {\n\t\t\tcontinue\n\t\t}\n\t\tnr, nc := r+dr[k], c+dc[k]\n\t\tn := nr*9 + nc\n\t\tif n != op {\n\t\t\tout = append(out, uint8(n))\n\t\t\tcontinue\n\t\t}\n\t\t// Соперник рядом: прямой прыжок, если за ним нет преграды.\n\t\tif b.CanStep(nr, nc, k) {\n\t\t\tout = append(out, uint8((nr+dr[k])*9+nc+dc[k]))\n\t\t\tcontinue\n\t\t}\n\t\t// Иначе боковые прыжки.\n\t\tfor _, s := range sideDirs[k] {\n\t\t\tif b.CanStep(nr, nc, s) {\n\t\t\t\tout = append(out, uint8((nr+dr[s])*9+nc+dc[s]))\n\t\t\t}\n\t\t}\n\t}\n\treturn out\n}\n\n// WithWall возвращает позицию после стены игрока, чей ход, и ok=false, если стена нелегальна.\n// Это медленная эталонная проверка (полный BFS обоих игроков): используй её для тестов,\n// а в поиске добавь правило "≥2 контактов" и ленивую легальность.\nfunc (b Board) WithWall(horizontal bool, r, c int) (Board, bool) {\n\torig := b\n\tif r < 0 || r > 7 || c < 0 || c > 7 || b.Walls[b.Turn] == 0 {\n\t\treturn orig, false\n\t}\n\tbit := uint64(1) << uint(r*8+c)\n\tif horizontal {\n\t\tif has(b.HW, r, c) || has(b.HW, r, c-1) || has(b.HW, r, c+1) || has(b.VW, r, c) {\n\t\t\treturn orig, false\n\t\t}\n\t\tb.HW |= bit\n\t} else {\n\t\tif has(b.VW, r, c) || has(b.VW, r-1, c) || has(b.VW, r+1, c) || has(b.HW, r, c) {\n\t\t\treturn orig, false\n\t\t}\n\t\tb.VW |= bit\n\t}\n\tb.Walls[b.Turn]--\n\td0 := b.Dist(GoalRow(0))\n\td1 := b.Dist(GoalRow(1))\n\tif d0[b.P[0]] < 0 || d1[b.P[1]] < 0 {\n\t\treturn orig, false\n\t}\n\treturn b, true\n}\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n')

os.makedirs('board', exist_ok=True)
with open('board/board_test.go', 'w', encoding='utf-8') as f:
    f.write('package board\n\nimport (\n    "testing"\n)\n\nfunc TestStartDistances(t *testing.T) {\n\tb := New()\n\tif d := b.Dist(8)[b.P[0]]; d != 8 {\n\t\tt.Fatalf("player0 dist = %d, want 8", d)\n\t}\n\tif d := b.Dist(0)[b.P[1]]; d != 8 {\n\t\tt.Fatalf("player1 dist = %d, want 8", d)\n\t}\n}\n\nfunc TestStartPawnMoves(t *testing.T) {\n\tb := New()\n\tm := b.PawnMoves(0, make([]uint8, 0, 8))\n\tif len(m) != 3 { // из верхнего края: вниз, влево, вправо\n\t\tt.Fatalf("got %d moves, want 3", len(m))\n\t}\n}\n\nfunc TestStraightJump(t *testing.T) {\n\tb := New()\n\tb.P = [2]uint8{4*9 + 4, 5*9 + 4} // соперник прямо под нами\n\tfound := false\n\tfor _, m := range b.PawnMoves(0, make([]uint8, 0, 8)) {\n\t\tif int(m) == 6*9+4 {\n\t\t\tfound = true\n\t\t}\n\t}\n\tif !found {\n\t\tt.Fatal("straight jump missing")\n\t}\n}\n\nfunc TestSideJumpWhenBlocked(t *testing.T) {\n\tb := New()\n\tb.P = [2]uint8{4*9 + 4, 5*9 + 4}\n\t// стена под соперником (между строками 5 и 6, столбцы 4-5)\n\tb.HW |= 1 << uint(5*8+4)\n\tmoves := b.PawnMoves(0, make([]uint8, 0, 8))\n\tleft, right, straight := false, false, false\n\tfor _, m := range moves {\n\t\tswitch int(m) {\n\t\tcase 5*9 + 3:\n\t\t\tleft = true\n\t\tcase 5*9 + 5:\n\t\t\tright = true\n\t\tcase 6*9 + 4:\n\t\t\tstraight = true\n\t\t}\n\t}\n\tif straight || !left || !right {\n\t\tt.Fatalf("bad jump set: %v", moves)\n\t}\n}\n\nfunc TestWallOverlapAndCross(t *testing.T) {\n\tb := New()\n\tb2, ok := b.WithWall(true, 3, 3)\n\tif !ok {\n\t\tt.Fatal("first wall must be legal")\n\t}\n\tif _, ok := b2.WithWall(true, 3, 4); ok {\n\t\tt.Fatal("overlapping wall must be illegal")\n\t}\n\tif _, ok := b2.WithWall(false, 3, 3); ok {\n\t\tt.Fatal("crossing wall must be illegal")\n\t}\n}\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n')

os.makedirs('path', exist_ok=True)
with open('path/path_test.go', 'w', encoding='utf-8') as f:
    f.write('package path\n\nimport (\n\t"math/rand"\n\t"testing"\n\n\t"quoridor/board"\n)\n\n// ---- независимый эталон: рёбра проверяются геометрически по списку стен ----\n\ntype wall struct {\n\th    bool\n\tr, c int\n}\n\nfunc decode(hw, vw uint64) (ws []wall) {\n\tfor i := 0; i < 64; i++ {\n\t\tif hw>>uint(i)&1 != 0 {\n\t\t\tws = append(ws, wall{true, i / 8, i % 8})\n\t\t}\n\t\tif vw>>uint(i)&1 != 0 {\n\t\t\tws = append(ws, wall{false, i / 8, i % 8})\n\t\t}\n\t}\n\treturn\n}\n\nfunc blocked(ws []wall, r1, c1, r2, c2 int) bool {\n\tfor _, w := range ws {\n\t\tif r1 != r2 {\n\t\t\tlo := r1\n\t\t\tif r2 < lo {\n\t\t\t\tlo = r2\n\t\t\t}\n\t\t\tif w.h && w.r == lo && (c1 == w.c || c1 == w.c+1) {\n\t\t\t\treturn true\n\t\t\t}\n\t\t} else {\n\t\t\tlo := c1\n\t\t\tif c2 < lo {\n\t\t\t\tlo = c2\n\t\t\t}\n\t\t\tif !w.h && w.c == lo && (r1 == w.r || r1 == w.r+1) {\n\t\t\t\treturn true\n\t\t\t}\n\t\t}\n\t}\n\treturn false\n}\n\nfunc refDist(hw, vw uint64, goal int) (d [81]int8) {\n\tws := decode(hw, vw)\n\tfor i := range d {\n\t\td[i] = -1\n\t}\n\tvar q []int\n\tfor c := 0; c < 9; c++ {\n\t\td[goal*9+c] = 0\n\t\tq = append(q, goal*9+c)\n\t}\n\tfor len(q) > 0 {\n\t\tcur := q[0]\n\t\tq = q[1:]\n\t\tr, c := cur/9, cur%9\n\t\tfor _, dd := range [4][2]int{{1, 0}, {-1, 0}, {0, 1}, {0, -1}} {\n\t\t\tnr, nc := r+dd[0], c+dd[1]\n\t\t\tif nr < 0 || nr > 8 || nc < 0 || nc > 8 || blocked(ws, r, c, nr, nc) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tif d[nr*9+nc] < 0 {\n\t\t\t\td[nr*9+nc] = d[cur] + 1\n\t\t\t\tq = append(q, nr*9+nc)\n\t\t\t}\n\t\t}\n\t}\n\treturn\n}\n\n// ---- генераторы позиций ----\n\n// Позиции из случайных легальных партий (прыжки, стены, разные стадии).\nfunc gamePositions(n int, seed int64) []board.Board {\n\trng := rand.New(rand.NewSource(seed))\n\tvar out []board.Board\n\tfor len(out) < n {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 100 && len(out) < n && b.Winner() < 0; ply++ {\n\t\t\tout = append(out, b)\n\t\t\tif rng.Intn(100) < 45 && b.Walls[b.Turn] > 0 {\n\t\t\t\tmoved := false\n\t\t\t\tfor try := 0; try < 30 && !moved; try++ {\n\t\t\t\t\th := rng.Intn(2) == 0\n\t\t\t\t\tr, c := rng.Intn(8), rng.Intn(8)\n\t\t\t\t\tif _, ok := b.WithWall(h, r, c); ok {\n\t\t\t\t\t\tb = b.Apply(board.WallMove(h, r, c))\n\t\t\t\t\t\tmoved = true\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\tif moved {\n\t\t\t\t\tcontinue\n\t\t\t\t}\n\t\t\t}\n\t\t\tms := b.PawnMoves(int(b.Turn), make([]uint8, 0, 8))\n\t\t\tb = b.Apply(board.PawnMove(int(ms[rng.Intn(len(ms))])))\n\t\t}\n\t}\n\treturn out\n}\n\nfunc TestFieldVsReferenceOnGames(t *testing.T) {\n\tfor i, b := range gamePositions(10000, 1) {\n\t\tfor p := 0; p < 2; p++ {\n\t\t\tg := board.GoalRow(p)\n\t\t\tgot, want := Field(&b, g), refDist(b.HW, b.VW, g)\n\t\t\tif got != want {\n\t\t\t\tt.Fatalf("pos %d goal %d: Field != ref\\n%+v", i, g, b)\n\t\t\t}\n\t\t\tif old := b.Dist(g); old != got {\n\t\t\t\tt.Fatalf("pos %d goal %d: Field != board.Dist", i, g)\n\t\t\t}\n\t\t\tif l := Len(&b, p); l != int(want[b.P[p]]) {\n\t\t\t\tt.Fatalf("pos %d p%d: Len=%d want %d", i, p, l, want[b.P[p]])\n\t\t\t}\n\t\t}\n\t}\n}\n\n// Сырые случайные маски (стены могут пересекаться и отрезать части поля): проверяем -1.\nfunc TestFieldVsReferenceRandomMasks(t *testing.T) {\n\trng := rand.New(rand.NewSource(2))\n\tunreachable := 0\n\tfor i := 0; i < 10000; i++ {\n\t\tvar b board.Board\n\t\tfor k := rng.Intn(40); k > 0; k-- {\n\t\t\tif rng.Intn(2) == 0 {\n\t\t\t\tb.HW |= 1 << uint(rng.Intn(64))\n\t\t\t} else {\n\t\t\t\tb.VW |= 1 << uint(rng.Intn(64))\n\t\t\t}\n\t\t}\n\t\tb.P = [2]uint8{uint8(rng.Intn(81)), uint8(rng.Intn(81))}\n\t\tfor p := 0; p < 2; p++ {\n\t\t\tg := board.GoalRow(p)\n\t\t\tgot, want := Field(&b, g), refDist(b.HW, b.VW, g)\n\t\t\tif got != want {\n\t\t\t\tt.Fatalf("mask %d goal %d: Field != ref (HW=%#x VW=%#x)", i, g, b.HW, b.VW)\n\t\t\t}\n\t\t\tif Len(&b, p) != int(want[b.P[p]]) {\n\t\t\t\tt.Fatalf("mask %d p%d: Len mismatch", i, p)\n\t\t\t}\n\t\t\tif want[b.P[p]] < 0 {\n\t\t\t\tunreachable++\n\t\t\t}\n\t\t}\n\t}\n\tt.Logf("unreachable pawn cases covered: %d", unreachable)\n}\n\nfunc TestStart(t *testing.T) {\n\tb := board.New()\n\tif Len(&b, 0) != 8 || Len(&b, 1) != 8 {\n\t\tt.Fatal("start Len must be 8")\n\t}\n}\n\n// ---- бенчмарки ----\n\nvar sink int8\nvar sinkI int\n\nfunc BenchmarkFieldStart(bm *testing.B) {\n\tb := board.New()\n\tfor i := 0; i < bm.N; i++ {\n\t\td := Field(&b, 8)\n\t\tsink = d[4]\n\t}\n}\n\nfunc BenchmarkFieldMid(bm *testing.B) {\n\tps := gamePositions(1000, 3)\n\tbm.ResetTimer()\n\tfor i := 0; i < bm.N; i++ {\n\t\td := Field(&ps[i%len(ps)], 8)\n\t\tsink = d[4]\n\t}\n}\n\nfunc BenchmarkLenMid(bm *testing.B) {\n\tps := gamePositions(1000, 3)\n\tbm.ResetTimer()\n\tfor i := 0; i < bm.N; i++ {\n\t\tsinkI = Len(&ps[i%len(ps)], 0)\n\t}\n}\n\nfunc BenchmarkBoardDistMid(bm *testing.B) { // старая реализация для сравнения\n\tps := gamePositions(1000, 3)\n\tbm.ResetTimer()\n\tfor i := 0; i < bm.N; i++ {\n\t\td := ps[i%len(ps)].Dist(8)\n\t\tsink = d[4]\n\t}\n}\n\n')

os.makedirs('path', exist_ok=True)
with open('path/path.go', 'w', encoding='utf-8') as f:
    f.write('// Package path: быстрые поля расстояний (BFS на битовых рядах), без аллокаций.\n//\n// Доска 9x9 хранится как [9]uint16: бит c в ряду r = клетка (r,c).\n// Из масок стен считаются маски заблокированных рёбер, затем BFS идёт слоями:\n// на каждом слое все ряды расширяются побитовыми операциями.\npackage path\n\nimport (\n\t"math/bits"\n\n\t"quoridor/board"\n)\n\nconst row9 = 0x1FF\n\ntype masks struct {\n\thb [8]uint16 // hb[r], бит c: ребро (r,c)-(r+1,c) заблокировано\n\tvb [9]uint16 // vb[r], бит c: ребро (r,c)-(r,c+1) заблокировано (c<8)\n}\n\nfunc build(b *board.Board) (m masks) {\n\tvar v [8]uint16\n\tfor r := 0; r < 8; r++ {\n\t\th := uint16((b.HW >> (uint(r) * 8)) & 0xFF) // стена c закрывает столбцы c и c+1\n\t\tm.hb[r] = h | h<<1\n\t\tv[r] = uint16((b.VW >> (uint(r) * 8)) & 0xFF) // стена r закрывает строки r и r+1\n\t}\n\tfor r := 0; r < 9; r++ {\n\t\tvar x uint16\n\t\tif r < 8 {\n\t\t\tx |= v[r]\n\t\t}\n\t\tif r > 0 {\n\t\t\tx |= v[r-1]\n\t\t}\n\t\tm.vb[r] = x\n\t}\n\treturn\n}\n\n// step считает новый слой по текущему фронту f, не заходя в vis.\nfunc (m *masks) step(f, vis *[9]uint16) (nf [9]uint16, any uint16) {\n\tfor r := 0; r < 9; r++ {\n\t\tfr := f[r]\n\t\tx := ((fr&^m.vb[r])<<1)&row9 | (fr&^(m.vb[r]<<1))>>1\n\t\tif r > 0 {\n\t\t\tx |= f[r-1] &^ m.hb[r-1]\n\t\t}\n\t\tif r < 8 {\n\t\t\tx |= f[r+1] &^ m.hb[r]\n\t\t}\n\t\tx &^= vis[r]\n\t\tnf[r] = x\n\t\tany |= x\n\t}\n\treturn\n}\n\n// Field: поле расстояний до строки goalRow, как board.Dist (-1 = недостижимо).\nfunc Field(b *board.Board, goalRow int) [81]int8 {\n\tm := build(b)\n\tvar d [81]int8\n\tfor i := range d {\n\t\td[i] = -1\n\t}\n\tvar vis, f [9]uint16\n\tf[goalRow], vis[goalRow] = row9, row9\n\tfor c := 0; c < 9; c++ {\n\t\td[goalRow*9+c] = 0\n\t}\n\tfor lvl := int8(1); ; lvl++ {\n\t\tnf, any := m.step(&f, &vis)\n\t\tif any == 0 {\n\t\t\treturn d\n\t\t}\n\t\tfor r := 0; r < 9; r++ {\n\t\t\tvis[r] |= nf[r]\n\t\t\tfor y := nf[r]; y != 0; y &= y - 1 {\n\t\t\t\td[r*9+bits.TrailingZeros16(y)] = lvl\n\t\t\t}\n\t\t}\n\t\tf = nf\n\t}\n}\n\n// Len: длина кратчайшего пути пешки игрока p до его цели (-1 = пути нет).\n// Останавливается, как только дошёл до пешки, поэтому быстрее полного поля.\nfunc Len(b *board.Board, p int) int {\n\tm := build(b)\n\tpr, pc := int(b.P[p])/9, uint(b.P[p])%9\n\tgoal := board.GoalRow(p)\n\tif pr == goal {\n\t\treturn 0\n\t}\n\tvar vis, f [9]uint16\n\tf[goal], vis[goal] = row9, row9\n\tfor lvl := 1; ; lvl++ {\n\t\tnf, any := m.step(&f, &vis)\n\t\tif any == 0 {\n\t\t\treturn -1\n\t\t}\n\t\tif nf[pr]>>pc&1 != 0 {\n\t\t\treturn lvl\n\t\t}\n\t\tfor r := 0; r < 9; r++ {\n\t\t\tvis[r] |= nf[r]\n\t\t}\n\t\tf = nf\n\t}\n}\n\n')

os.makedirs('eval', exist_ok=True)
with open('eval/eval.go', 'w', encoding='utf-8') as f:
    f.write('// Package eval: оценочные функции. Оценка всегда с точки зрения того, чей ход (negamax).\npackage eval\n\nimport (\n\t"quoridor/board"\n\t"quoridor/path"\n)\n\n// V1: гонка путей + стены в запасе + право хода (этап 5).\n//\n//\t100 за каждый шаг преимущества в пути, 140 за стену в запасе, 20 за право хода.\n//\n// Вес стены подобран матчами против Walker: при 10–40 глубокий поиск тратит все стены\n// и от страха перед стенами соперника топчется на месте (проигрывает «ходоку»),\n// при 120–250 бот стабильно выигрывает на любой глубине 3–6. Точный вес потом уточнит SPSA.\nfunc V1(b *board.Board) int {\n\tme := int(b.Turn)\n\tdMe, dOp := path.Len(b, me), path.Len(b, 1-me)\n\treturn 100*(dOp-dMe) + 140*(int(b.Walls[me])-int(b.Walls[1-me])) + 20\n}\n\n')

os.makedirs('eval', exist_ok=True)
with open('eval/eval_test.go', 'w', encoding='utf-8') as f:
    f.write('package eval\n\nimport (\n\t"math/rand"\n\t"testing"\n\n\t"quoridor/board"\n\t"quoridor/movegen"\n)\n\n// mirror поворачивает доску на 180° и меняет игроков: позиция «глазами соперника».\nfunc mirror(b board.Board) board.Board {\n\tvar m board.Board\n\tfor i := 0; i < 64; i++ {\n\t\tr, c := i/8, i%8\n\t\tj := (7-r)*8 + (7 - c)\n\t\tif b.HW>>uint(i)&1 != 0 {\n\t\t\tm.HW |= 1 << uint(j)\n\t\t}\n\t\tif b.VW>>uint(i)&1 != 0 {\n\t\t\tm.VW |= 1 << uint(j)\n\t\t}\n\t}\n\tfor p := 0; p < 2; p++ {\n\t\tr, c := int(b.P[p])/9, int(b.P[p])%9\n\t\tm.P[1-p] = uint8((8-r)*9 + (8 - c))\n\t\tm.Walls[1-p] = b.Walls[p]\n\t}\n\tm.Turn = 1 - b.Turn\n\treturn m\n}\n\nfunc TestSymmetry(t *testing.T) {\n\trng := rand.New(rand.NewSource(1))\n\tn := 0\n\tfor g := 0; g < 100; g++ {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 60 && b.Winner() < 0; ply++ {\n\t\t\tif V1(&b) != V1(ptr(mirror(b))) {\n\t\t\t\tt.Fatalf("asymmetric eval: %d vs %d\\n%+v", V1(&b), V1(ptr(mirror(b))), b)\n\t\t\t}\n\t\t\tn++\n\t\t\tms := movegen.Gen(&b, nil)\n\t\t\tb = b.Apply(ms[rng.Intn(len(ms))])\n\t\t}\n\t}\n\tt.Logf("checked %d positions", n)\n}\n\nfunc ptr(b board.Board) *board.Board { return &b }\n\nfunc TestStart(t *testing.T) {\n\tb := board.New()\n\tif got := V1(&b); got != 20 {\n\t\tt.Fatalf("start eval = %d, want 20 (tempo only)", got)\n\t}\n}\n\n')

os.makedirs('eval', exist_ok=True)
with open('eval/v2.go', 'w', encoding='utf-8') as f:
    f.write('package eval\n\nimport (\n\t"fmt"\n\t"math"\n\t"strconv"\n\t"strings"\n\n\t"quoridor/board"\n\t"quoridor/path"\n)\n\n// Params: веса оценки v2. Все признаки считаются для обоих игроков и вычитаются («я минус соперник»).\ntype Params struct {\n\tD           int // шаг разницы путей\n\tTempo       int // право хода\n\tFlex        int // гибкость: на каждые 1/4 бита log2(числа кратчайших путей)\n\tCut         int // узкое место: слой DAG из одной клетки (стеной можно удлинить маршрут)\n\tExit        int // «неудлиняемое» узкое ребро (покрыть его легальной стеной нельзя)\n\tWalls       int // стена в запасе\n\tWallsNarrow int // добавка к весу стены за каждое узкое место на путях обоих игроков\n}\n\n// DefaultV2: веса после SPSA (300 итераций self-play, глубина 3, cmd/tune), стартуя с ручных\n// Tempo 20, Flex 3, Cut 0, Exit 20, Walls 170, WallsNarrow 3.\n// Проверка против ручных весов (глубина 4, SPSA тренировался на глубине 3):\n// 400 партий +61 Elo (H1), ещё 600 партий +26 Elo [-1, 54]; на глубине 3 разницы нет (+6, 1000 партий).\n// Вес стены остался прежним (~170): SPSA подтвердил ручной подбор.\nfunc DefaultV2() Params {\n\treturn Params{D: 100, Tempo: 2, Flex: 0, Cut: 9, Exit: 42, Walls: 171, WallsNarrow: 0}\n}\n\ntype side struct {\n\td     int // длина кратчайшего пути пешки до цели\n\tlogp  int // 4*log2(число кратчайших путей), с насыщением\n\tcut   int // слои DAG шириной 1 (не считая стартовой и целевой клеток)\n\texit  int // одиночные рёбра между слоями, которые нельзя перекрыть ни одной стеной\n\twidth int // максимальная ширина слоя\n}\n\nconst pathCap = 1 << 20\n\nvar dr = [4]int{-1, 1, 0, 0}\nvar dc = [4]int{0, 0, -1, 1}\n\nfunc has(m uint64, r, c int) bool {\n\tif r < 0 || r > 7 || c < 0 || c > 7 {\n\t\treturn false\n\t}\n\treturn m&(1<<uint(r*8+c)) != 0\n}\n\n// blockable: можно ли поставить хоть одну стену (по перекрытию/кресту, без учёта запаса и путей),\n// перекрывающую шаг из (r,c) в направлении k.\nfunc blockable(b *board.Board, r, c, k int) bool {\n\ttry := func(h bool, wr, wc int) bool {\n\t\tif wr < 0 || wr > 7 || wc < 0 || wc > 7 {\n\t\t\treturn false\n\t\t}\n\t\tif h {\n\t\t\treturn !(has(b.HW, wr, wc) || has(b.HW, wr, wc-1) || has(b.HW, wr, wc+1) || has(b.VW, wr, wc))\n\t\t}\n\t\treturn !(has(b.VW, wr, wc) || has(b.VW, wr-1, wc) || has(b.VW, wr+1, wc) || has(b.HW, wr, wc))\n\t}\n\tswitch k {\n\tcase board.Up:\n\t\treturn try(true, r-1, c) || try(true, r-1, c-1)\n\tcase board.Down:\n\t\treturn try(true, r, c) || try(true, r, c-1)\n\tcase board.Left:\n\t\treturn try(false, r, c-1) || try(false, r-1, c-1)\n\tdefault:\n\t\treturn try(false, r, c) || try(false, r-1, c)\n\t}\n}\n\n// analyze строит DAG кратчайших путей пешки игрока p (рёбра, где расстояние до цели убывает на 1).\nfunc analyze(b *board.Board, p int) side {\n\tf := path.Field(b, board.GoalRow(p))\n\tstart := int(b.P[p])\n\tD := int(f[start])\n\tvar s side\n\ts.d = D\n\tif D <= 0 {\n\t\ts.logp = 0\n\t\treturn s\n\t}\n\tvar paths [81]uint32\n\tvar seen [81]bool\n\tvar q [81]uint8\n\tvar layer [82]uint8\n\tvar edges [82]uint8\n\tvar lastCur [82]uint8\n\tvar lastDir [82]uint8\n\tpaths[start], seen[start] = 1, true\n\tq[0] = uint8(start)\n\th, t := 0, 1\n\tfor h < t {\n\t\tcur := int(q[h])\n\t\th++\n\t\tr, c := cur/9, cur%9\n\t\tdcur := int(f[cur])\n\t\tlayer[dcur]++\n\t\tif dcur == 0 {\n\t\t\tcontinue\n\t\t}\n\t\tfor k := 0; k < 4; k++ {\n\t\t\tif !b.CanStep(r, c, k) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tn := (r+dr[k])*9 + c + dc[k]\n\t\t\tif int(f[n]) != dcur-1 {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tif edges[dcur] == 0 {\n\t\t\t\tlastCur[dcur], lastDir[dcur] = uint8(cur), uint8(k)\n\t\t\t}\n\t\t\tedges[dcur]++\n\t\t\tif pv := paths[n] + paths[cur]; pv > pathCap {\n\t\t\t\tpaths[n] = pathCap\n\t\t\t} else {\n\t\t\t\tpaths[n] = pv\n\t\t\t}\n\t\t\tif !seen[n] {\n\t\t\t\tseen[n] = true\n\t\t\t\tq[t] = uint8(n)\n\t\t\t\tt++\n\t\t\t}\n\t\t}\n\t}\n\ttotal := uint32(0)\n\tfor c := 0; c < 9; c++ {\n\t\ttotal += paths[board.GoalRow(p)*9+c]\n\t}\n\tif total > pathCap {\n\t\ttotal = pathCap\n\t}\n\ts.logp = int(4 * math.Log2(float64(total)))\n\tfor d := 1; d <= D; d++ {\n\t\tif int(layer[d]) > s.width {\n\t\t\ts.width = int(layer[d])\n\t\t}\n\t\tif d < D && layer[d] == 1 {\n\t\t\ts.cut++\n\t\t}\n\t\tif edges[d] == 1 {\n\t\t\tcur := int(lastCur[d])\n\t\t\tif !blockable(b, cur/9, cur%9, int(lastDir[d])) {\n\t\t\t\ts.exit++\n\t\t\t}\n\t\t}\n\t}\n\treturn s\n}\n\n// Eval: оценка v2 с точки зрения того, чей ход.\nfunc (p *Params) Eval(b board.Board) int {\n\tme := int(b.Turn)\n\twMe, wOp := int(b.Walls[me]), int(b.Walls[1-me])\n\tif wMe == 0 && wOp == 0 { // чистая гонка: точный исход (с точностью до блокировок пешками)\n\t\tdMe, dOp := path.Len(&b, me), path.Len(&b, 1-me)\n\t\tif dMe <= dOp {\n\t\t\treturn 10000 - 10*dMe\n\t\t}\n\t\treturn -(10000 - 10*dOp)\n\t}\n\tam, ao := analyze(&b, me), analyze(&b, 1-me)\n\tv := p.D*(ao.d-am.d) + p.Tempo\n\tv += p.Flex * (am.logp - ao.logp)\n\tcutOp, cutMe := 0, 0\n\tif wMe > 0 {\n\t\tcutOp = ao.cut // мои удары по его узким местам\n\t}\n\tif wOp > 0 {\n\t\tcutMe = am.cut // его удары по моим\n\t}\n\tv += p.Cut * (cutOp - cutMe)\n\tv += p.Exit * (am.exit - ao.exit)\n\tv += (p.Walls + p.WallsNarrow*(am.cut+ao.cut)) * (wMe - wOp)\n\treturn v\n}\n\n// Set задаёт параметр по имени (для командной строки и SPSA).\nfunc (p *Params) Set(name string, v int) bool {\n\tswitch name {\n\tcase "D":\n\t\tp.D = v\n\tcase "Tempo":\n\t\tp.Tempo = v\n\tcase "Flex":\n\t\tp.Flex = v\n\tcase "Cut":\n\t\tp.Cut = v\n\tcase "Exit":\n\t\tp.Exit = v\n\tcase "Walls":\n\t\tp.Walls = v\n\tcase "WallsNarrow":\n\t\tp.WallsNarrow = v\n\tdefault:\n\t\treturn false\n\t}\n\treturn true\n}\n\nfunc (p Params) String() string {\n\treturn fmt.Sprintf("Params{D: %d, Tempo: %d, Flex: %d, Cut: %d, Exit: %d, Walls: %d, WallsNarrow: %d}",\n\t\tp.D, p.Tempo, p.Flex, p.Cut, p.Exit, p.Walls, p.WallsNarrow)\n}\n\n// ParseParams: "Flex=5,Walls=150" поверх DefaultV2.\nfunc ParseParams(spec string) (Params, error) {\n\tp := DefaultV2()\n\tif strings.TrimSpace(spec) == "" {\n\t\treturn p, nil\n\t}\n\tfor _, kv := range strings.Split(spec, ",") {\n\t\tparts := strings.SplitN(strings.TrimSpace(kv), "=", 2)\n\t\tif len(parts) != 2 {\n\t\t\treturn p, fmt.Errorf("bad param %q (want Name=Value)", kv)\n\t\t}\n\t\tv, err := strconv.Atoi(parts[1])\n\t\tif err != nil || !p.Set(parts[0], v) {\n\t\t\treturn p, fmt.Errorf("bad param %q", kv)\n\t\t}\n\t}\n\treturn p, nil\n}\n\n')

os.makedirs('eval', exist_ok=True)
with open('eval/v2_test.go', 'w', encoding='utf-8') as f:
    f.write('package eval\n\nimport (\n\t"math"\n\t"math/rand"\n\t"testing"\n\n\t"quoridor/board"\n\t"quoridor/movegen"\n)\n\nfunc TestV2Symmetry(t *testing.T) {\n\tp := DefaultV2()\n\trng := rand.New(rand.NewSource(3))\n\tfor g := 0; g < 150; g++ {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 80 && b.Winner() < 0; ply++ {\n\t\t\tm := mirror(b)\n\t\t\tif p.Eval(b) != p.Eval(m) {\n\t\t\t\tt.Fatalf("asymmetric v2: %d vs %d\\n%+v", p.Eval(b), p.Eval(m), b)\n\t\t\t}\n\t\t\tms := movegen.Gen(&b, nil)\n\t\t\tb = b.Apply(ms[rng.Intn(len(ms))])\n\t\t}\n\t}\n}\n\nfunc TestV2Start(t *testing.T) {\n\tp := DefaultV2()\n\tb := board.New()\n\tif got := p.Eval(b); got != p.Tempo {\n\t\tt.Fatalf("start eval = %d, want tempo %d", got, p.Tempo)\n\t}\n}\n\n// ---- эталон признаков: перебор по определению ----\n\nfunc bfsFrom(b *board.Board, from int) (d [81]int) {\n\tfor i := range d {\n\t\td[i] = -1\n\t}\n\td[from] = 0\n\tq := []int{from}\n\tdr := [4]int{-1, 1, 0, 0}\n\tdc := [4]int{0, 0, -1, 1}\n\tfor len(q) > 0 {\n\t\tcur := q[0]\n\t\tq = q[1:]\n\t\tfor k := 0; k < 4; k++ {\n\t\t\tif !b.CanStep(cur/9, cur%9, k) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tn := (cur/9+dr[k])*9 + cur%9 + dc[k]\n\t\t\tif d[n] < 0 {\n\t\t\t\td[n] = d[cur] + 1\n\t\t\t\tq = append(q, n)\n\t\t\t}\n\t\t}\n\t}\n\treturn\n}\n\n// refAnalyze: слой = множество клеток x, лежащих на кратчайшем пути (dStart[x]+dGoal[x]==D).\nfunc refAnalyze(b *board.Board, p int) (D, paths, cut, width int) {\n\tstart := int(b.P[p])\n\tds := bfsFrom(b, start)\n\tgoal := board.GoalRow(p)\n\t// расстояние до цели: мультиисточник — через board.Dist (старая независимая реализация)\n\tdg := b.Dist(goal)\n\tD = int(dg[start])\n\tif D <= 0 {\n\t\treturn\n\t}\n\tlayer := make([]int, D+1)\n\tfor x := 0; x < 81; x++ {\n\t\tif ds[x] >= 0 && int(dg[x]) >= 0 && ds[x]+int(dg[x]) == D {\n\t\t\tlayer[ds[x]]++\n\t\t}\n\t}\n\tfor i := 0; i <= D; i++ {\n\t\tif layer[i] > width {\n\t\t\twidth = layer[i]\n\t\t}\n\t\tif i >= 1 && i < D && layer[i] == 1 {\n\t\t\tcut++\n\t\t}\n\t}\n\t// число путей перебором\n\tvar rec func(cur int) int\n\tdr := [4]int{-1, 1, 0, 0}\n\tdc := [4]int{0, 0, -1, 1}\n\trec = func(cur int) int {\n\t\tif int(dg[cur]) == 0 {\n\t\t\treturn 1\n\t\t}\n\t\tn := 0\n\t\tfor k := 0; k < 4; k++ {\n\t\t\tif !b.CanStep(cur/9, cur%9, k) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tnx := (cur/9+dr[k])*9 + cur%9 + dc[k]\n\t\t\tif dg[nx] == dg[cur]-1 {\n\t\t\t\tn += rec(nx)\n\t\t\t}\n\t\t}\n\t\treturn n\n\t}\n\tpaths = rec(start)\n\treturn\n}\n\nfunc TestAnalyzeMatchesBruteForce(t *testing.T) {\n\trng := rand.New(rand.NewSource(4))\n\tchecked := 0\n\tfor g := 0; g < 120; g++ {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 80 && b.Winner() < 0; ply++ {\n\t\t\tfor p := 0; p < 2; p++ {\n\t\t\t\tD, paths, cut, width := refAnalyze(&b, p)\n\t\t\t\tif paths > 200000 {\n\t\t\t\t\tcontinue // слишком много путей для перебора\n\t\t\t\t}\n\t\t\t\tgot := analyze(&b, p)\n\t\t\t\t// число путей: сравниваем 4*log2 с точностью до округления\n\t\t\t\twantLog := 0\n\t\t\t\tif paths > 0 {\n\t\t\t\t\twantLog = int(4 * log2(float64(paths)))\n\t\t\t\t}\n\t\t\t\tif got.d != D || got.cut != cut || got.width != width || got.logp != wantLog {\n\t\t\t\t\tt.Fatalf("p%d: got %+v want d=%d cut=%d width=%d logp=%d (paths=%d)\\n%+v", p, got, D, cut, width, wantLog, paths, b)\n\t\t\t\t}\n\t\t\t\tchecked++\n\t\t\t}\n\t\t\tms := movegen.Gen(&b, nil)\n\t\t\tb = b.Apply(ms[rng.Intn(len(ms))])\n\t\t}\n\t}\n\tt.Logf("checked %d analyses", checked)\n}\n\nfunc BenchmarkV2(bm *testing.B) {\n\tp := DefaultV2()\n\trng := rand.New(rand.NewSource(5))\n\tvar ps []board.Board\n\tb := board.New()\n\tfor len(ps) < 500 {\n\t\tps = append(ps, b)\n\t\tms := movegen.Gen(&b, nil)\n\t\tb = b.Apply(ms[rng.Intn(len(ms))])\n\t\tif b.Winner() >= 0 {\n\t\t\tb = board.New()\n\t\t}\n\t}\n\tbm.ResetTimer()\n\ts := 0\n\tfor i := 0; i < bm.N; i++ {\n\t\ts += p.Eval(ps[i%len(ps)])\n\t}\n\t_ = s\n}\n\nfunc BenchmarkV1(bm *testing.B) {\n\trng := rand.New(rand.NewSource(5))\n\tvar ps []board.Board\n\tb := board.New()\n\tfor len(ps) < 500 {\n\t\tps = append(ps, b)\n\t\tms := movegen.Gen(&b, nil)\n\t\tb = b.Apply(ms[rng.Intn(len(ms))])\n\t\tif b.Winner() >= 0 {\n\t\t\tb = board.New()\n\t\t}\n\t}\n\tbm.ResetTimer()\n\ts := 0\n\tfor i := 0; i < bm.N; i++ {\n\t\ts += V1(&ps[i%len(ps)])\n\t}\n\t_ = s\n}\n\nfunc log2(x float64) float64 { return math.Log2(x) }\n\n')

os.makedirs('search', exist_ok=True)
with open('search/search.go', 'w', encoding='utf-8') as f:
    f.write('// Package search: negamax + alpha-beta + итеративное углубление + таблица транспозиций\n// + упорядочивание ходов (ход из TT, продвижение пешки, killer, history).\n//\n// В горячем цикле нет аллокаций: все буферы лежат в Searcher и создаются один раз в New.\npackage search\n\nimport (\n\t"math/bits"\n\t"time"\n\n\t"quoridor/board"\n\t"quoridor/eval"\n\t"quoridor/movegen"\n\t"quoridor/path"\n)\n\nconst (\n\tMate    = 30000\n\tMateThr = Mate - 1000\n\tInf     = Mate + 1\n\tmaxPly  = 64\n\tmaxMv   = 160\n)\n\nconst (\n\tflagExact = iota\n\tflagLower // score >= beta (отсечение)\n\tflagUpper // score <= alpha\n)\n\ntype entry struct { // 16 байт\n\tkey   uint64\n\tscore int16\n\tdepth int8\n\tflag  uint8\n\tmove  uint16\n}\n\ntype Limits struct {\n\tMaxDepth int           // 0 = без ограничения (до maxPly-4)\n\tTime     time.Duration // 0 = без ограничения\n}\n\ntype Result struct {\n\tMove    board.Move\n\tScore   int\n\tDepth   int // последняя полностью завершённая глубина\n\tNodes   uint64\n\tElapsed time.Duration\n}\n\ntype Searcher struct {\n\t// Eval принимает позицию по значению: так указатель не «утекает» в вызов по значению-функции\n\t// и позиции в рекурсии остаются на стеке (иначе аллокация на каждый узел).\n\tEval func(board.Board) int\n\n\t// ExtendWalls: продлевать ветку на 1 ход после стены, удлиняющей путь соперника на >=2\n\t// (не более ExtMax продлений на одной ветке; 0 = 2).\n\tExtendWalls bool\n\tExtMax      int8\n\n\ttt   []entry\n\tmask uint64\n\n\tmoves   [maxPly][maxMv]board.Move\n\tscores  [maxPly][maxMv]int32\n\tkeys    [maxPly + 1]uint64\n\textUsed [maxPly + 1]int8\n\tboards  [maxPly + 1]board.Board // позиции по глубине: так в рекурсии нет адресов локальных переменных\n\tkillers [maxPly][2]board.Move\n\thistory [2][576]int32\n\n\tnodes     uint64\n\tstop      bool\n\tdeadline  time.Time\n\trootBest  board.Move\n\trootScore int\n\trootDone  bool\n}\n\n// New создаёт поисковик с TT на 2^ttBits записей (16 байт каждая; 20 -> 16 МБ).\nfunc New(ttBits uint) *Searcher {\n\treturn &Searcher{\n\t\tEval: func(b board.Board) int { return eval.V1(&b) },\n\t\ttt:   make([]entry, 1<<ttBits),\n\t\tmask: 1<<ttBits - 1,\n\t}\n}\n\n// Clear: новая партия.\nfunc (s *Searcher) Clear() {\n\tfor i := range s.tt {\n\t\ts.tt[i] = entry{}\n\t}\n\ts.killers = [maxPly][2]board.Move{}\n\ts.history = [2][576]int32{}\n}\n\nfunc mix(x uint64) uint64 {\n\tx ^= x >> 30\n\tx *= 0xBF58476D1CE4E5B9\n\tx ^= x >> 27\n\tx *= 0x94D049BB133111EB\n\tx ^= x >> 31\n\treturn x\n}\n\n// Hash: ключ позиции (стены, пешки, запас стен, чей ход).\nfunc Hash(b *board.Board) uint64 {\n\tpack := uint64(b.P[0]) | uint64(b.P[1])<<8 | uint64(b.Walls[0])<<16 | uint64(b.Walls[1])<<24 | uint64(b.Turn)<<32\n\treturn mix(b.HW*0x9E3779B97F4A7C15 ^ bits.RotateLeft64(b.VW*0xC2B2AE3D27D4EB4F, 23) ^ mix(pack+0x632BE59BD9B4E019))\n}\n\n// Search ищет лучший ход. Не аллоцирует.\nfunc (s *Searcher) Search(b board.Board, lim Limits) Result {\n\tstart := time.Now()\n\ts.nodes, s.stop = 0, false\n\tif lim.Time > 0 {\n\t\ts.deadline = start.Add(lim.Time)\n\t} else {\n\t\ts.deadline = start.Add(24 * time.Hour)\n\t}\n\tmaxD := maxPly - 4\n\tif lim.MaxDepth > 0 && lim.MaxDepth < maxD {\n\t\tmaxD = lim.MaxDepth\n\t}\n\tvar res Result\n\tfor d := 1; d <= maxD; d++ {\n\t\ts.rootDone = false\n\t\ts.boards[0] = b\n\t\ts.extUsed[0] = 0\n\t\tscore := s.negamax(&s.boards[0], d, -Inf, Inf, 0)\n\t\tif s.stop {\n\t\t\tif s.rootDone { // ход, полностью проверенный на этой глубине, годится\n\t\t\t\tres.Move, res.Score = s.rootBest, s.rootScore\n\t\t\t}\n\t\t\tbreak\n\t\t}\n\t\tres.Move, res.Score, res.Depth = s.rootBest, score, d\n\t\tif score > MateThr || score < -MateThr {\n\t\t\tbreak\n\t\t}\n\t}\n\tres.Nodes, res.Elapsed = s.nodes, time.Since(start)\n\treturn res\n}\n\nfunc (s *Searcher) negamax(b *board.Board, depth, alpha, beta, ply int) int {\n\ts.nodes++\n\tif s.nodes&1023 == 0 && time.Now().After(s.deadline) {\n\t\ts.stop = true\n\t}\n\tif s.stop {\n\t\treturn 0\n\t}\n\tif b.Winner() >= 0 { // победитель — тот, кто только что походил\n\t\treturn -(Mate - ply)\n\t}\n\tkey := Hash(b)\n\tif ply > 0 {\n\t\tfor i := ply - 2; i >= 0; i -= 2 {\n\t\t\tif s.keys[i] == key {\n\t\t\t\treturn 0 // повторение позиции на пути: ничья\n\t\t\t}\n\t\t}\n\t}\n\ts.keys[ply] = key\n\tif depth <= 0 || ply >= maxPly-2 {\n\t\treturn s.Eval(*b)\n\t}\n\n\talpha0 := alpha\n\tvar ttMove board.Move\n\te := &s.tt[key&s.mask]\n\tif e.key == key {\n\t\tttMove = board.Move(e.move)\n\t\tif int(e.depth) >= depth && ply > 0 {\n\t\t\tsc := int(e.score)\n\t\t\tif sc > MateThr {\n\t\t\t\tsc -= ply\n\t\t\t} else if sc < -MateThr {\n\t\t\t\tsc += ply\n\t\t\t}\n\t\t\tswitch e.flag {\n\t\t\tcase flagExact:\n\t\t\t\treturn sc\n\t\t\tcase flagLower:\n\t\t\t\tif sc > alpha {\n\t\t\t\t\talpha = sc\n\t\t\t\t}\n\t\t\tcase flagUpper:\n\t\t\t\tif sc < beta {\n\t\t\t\t\tbeta = sc\n\t\t\t\t}\n\t\t\t}\n\t\t\tif alpha >= beta {\n\t\t\t\treturn sc\n\t\t\t}\n\t\t}\n\t}\n\n\tms := movegen.Candidates(b, s.moves[ply][:0])\n\tn := len(ms)\n\tsc := s.scores[ply][:n]\n\tturn := int(b.Turn)\n\tvar d [81]int8\n\thaveD := depth >= 2\n\tif haveD {\n\t\td = path.Field(b, board.GoalRow(turn))\n\t}\n\tcur := int(b.P[turn])\n\tfor i, m := range ms {\n\t\tvar v int32\n\t\tswitch {\n\t\tcase m == ttMove:\n\t\t\tv = 1 << 30\n\t\tcase m == s.killers[ply][0]:\n\t\t\tv = 1 << 21\n\t\tcase m == s.killers[ply][1]:\n\t\t\tv = 1<<21 - 1\n\t\tcase m.Kind() == board.KindPawn && haveD:\n\t\t\tv = 1<<22 + 100*int32(d[cur]-d[m.Idx()])\n\t\tdefault:\n\t\t\tv = s.history[turn][m]\n\t\t}\n\t\tsc[i] = v\n\t}\n\n\tbest := -Inf\n\tvar bestMove board.Move\n\ttried := 0\n\tdOpp0 := -1\n\tfor i := 0; i < n; i++ {\n\t\tbi := i // выбор лучшего по оценке\n\t\tfor j := i + 1; j < n; j++ {\n\t\t\tif sc[j] > sc[bi] {\n\t\t\t\tbi = j\n\t\t\t}\n\t\t}\n\t\tms[i], ms[bi] = ms[bi], ms[i]\n\t\tsc[i], sc[bi] = sc[bi], sc[i]\n\t\tm := ms[i]\n\t\tif !movegen.Legal(b, m) {\n\t\t\tcontinue\n\t\t}\n\t\ts.boards[ply+1] = b.Apply(m)\n\t\tnd := depth - 1\n\t\ts.extUsed[ply+1] = s.extUsed[ply]\n\t\tif s.ExtendWalls && depth >= 2 && m.Kind() != board.KindPawn {\n\t\t\tlim := s.ExtMax\n\t\t\tif lim == 0 {\n\t\t\t\tlim = 2\n\t\t\t}\n\t\t\tif s.extUsed[ply] < lim {\n\t\t\t\tif dOpp0 < 0 {\n\t\t\t\t\tdOpp0 = path.Len(b, 1-turn)\n\t\t\t\t}\n\t\t\t\tif path.Len(&s.boards[ply+1], 1-turn)-dOpp0 >= 2 {\n\t\t\t\t\tnd++\n\t\t\t\t\ts.extUsed[ply+1]++\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t\tscore := -s.negamax(&s.boards[ply+1], nd, -beta, -alpha, ply+1)\n\t\tif s.stop {\n\t\t\treturn 0\n\t\t}\n\t\ttried++\n\t\tif score > best {\n\t\t\tbest, bestMove = score, m\n\t\t\tif ply == 0 {\n\t\t\t\ts.rootBest, s.rootScore, s.rootDone = m, score, true\n\t\t\t}\n\t\t\tif score > alpha {\n\t\t\t\talpha = score\n\t\t\t\tif alpha >= beta {\n\t\t\t\t\tif s.killers[ply][0] != m {\n\t\t\t\t\t\ts.killers[ply][1], s.killers[ply][0] = s.killers[ply][0], m\n\t\t\t\t\t}\n\t\t\t\t\ts.history[turn][m] += int32(depth * depth)\n\t\t\t\t\tbreak\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t}\n\tif tried == 0 {\n\t\treturn s.Eval(*b)\n\t}\n\n\tflag := uint8(flagExact)\n\tif best <= alpha0 {\n\t\tflag = flagUpper\n\t} else if best >= beta {\n\t\tflag = flagLower\n\t}\n\tst := best\n\tif st > MateThr {\n\t\tst += ply\n\t} else if st < -MateThr {\n\t\tst -= ply\n\t}\n\t*e = entry{key: key, score: int16(st), depth: int8(depth), flag: flag, move: uint16(bestMove)}\n\treturn best\n}\n\n')

os.makedirs('search', exist_ok=True)
with open('search/search_test.go', 'w', encoding='utf-8') as f:
    f.write('package search\n\nimport (\n\t"math/rand"\n\t"testing"\n\t"time"\n\n\t"quoridor/board"\n\t"quoridor/eval"\n\t"quoridor/movegen"\n)\n\nfunc positions(n int, seed int64) []board.Board {\n\trng := rand.New(rand.NewSource(seed))\n\tvar out []board.Board\n\tfor len(out) < n {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 100 && len(out) < n && b.Winner() < 0; ply++ {\n\t\t\tout = append(out, b)\n\t\t\tms := movegen.Gen(&b, nil)\n\t\t\tb = b.Apply(ms[rng.Intn(len(ms))])\n\t\t}\n\t}\n\treturn out\n}\n\n// Эталон: обычный минимакс без отсечений и TT, с тем же правилом повторов.\nfunc refMinimax(b *board.Board, depth, ply int, keys *[maxPly + 1]uint64) int {\n\tif b.Winner() >= 0 {\n\t\treturn -(Mate - ply)\n\t}\n\tkey := Hash(b)\n\tif ply > 0 {\n\t\tfor i := ply - 2; i >= 0; i -= 2 {\n\t\t\tif keys[i] == key {\n\t\t\t\treturn 0\n\t\t\t}\n\t\t}\n\t}\n\tkeys[ply] = key\n\tif depth <= 0 {\n\t\treturn eval.V1(b)\n\t}\n\tms := movegen.Gen(b, nil)\n\tif len(ms) == 0 {\n\t\treturn eval.V1(b)\n\t}\n\tbest := -Inf\n\tfor _, m := range ms {\n\t\tnb := b.Apply(m)\n\t\tif v := -refMinimax(&nb, depth-1, ply+1, keys); v > best {\n\t\t\tbest = v\n\t\t}\n\t}\n\treturn best\n}\n\nfunc TestMatchesPlainMinimax(t *testing.T) {\n\tn, maxD := 60, 3\n\tif testing.Short() {\n\t\tn = 15\n\t}\n\ts := New(18)\n\tfor i, b := range positions(n, 1) {\n\t\tfor d := 1; d <= maxD; d++ {\n\t\t\ts.Clear()\n\t\t\tgot := s.Search(b, Limits{MaxDepth: d})\n\t\t\tvar keys [maxPly + 1]uint64\n\t\t\twant := refMinimax(&b, d, 0, &keys)\n\t\t\tif got.Score != want || got.Depth != d {\n\t\t\t\tt.Fatalf("pos %d depth %d: search=%d (depth %d) minimax=%d\\n%+v", i, d, got.Score, got.Depth, want, b)\n\t\t\t}\n\t\t\t// ход должен быть легальным\n\t\t\tif !inMoves(b.Moves(nil), got.Move) {\n\t\t\t\tt.Fatalf("pos %d depth %d: illegal best move %v", i, d, got.Move)\n\t\t\t}\n\t\t}\n\t}\n}\n\nfunc inMoves(ms []board.Move, m board.Move) bool {\n\tfor _, x := range ms {\n\t\tif x == m {\n\t\t\treturn true\n\t\t}\n\t}\n\treturn false\n}\n\nfunc TestTakesImmediateWin(t *testing.T) {\n\tb := board.New()\n\tb.P = [2]uint8{7*9 + 4, 8*9 + 0}\n\tr := New(16).Search(b, Limits{MaxDepth: 4})\n\tif r.Move != board.PawnMove(8*9+4) || r.Score != Mate-1 {\n\t\tt.Fatalf("got %+v", r)\n\t}\n}\n\nfunc TestAvoidsLosingRace(t *testing.T) {\n\t// Соперник на шаг от цели и его ход следующий: нужно либо выиграть сразу, либо поставить стену.\n\tb := board.New()\n\tb.P = [2]uint8{3*9 + 4, 1*9 + 4}\n\tb.Turn = 0\n\tr := New(18).Search(b, Limits{MaxDepth: 6})\n\tif r.Move.Kind() == board.KindPawn {\n\t\tt.Fatalf("must block with a wall, got pawn move %v (score %d)", r.Move, r.Score)\n\t}\n}\n\nfunc TestTimeLimit(t *testing.T) {\n\ts := New(20)\n\tstart := time.Now()\n\tr := s.Search(board.New(), Limits{Time: 200 * time.Millisecond})\n\tel := time.Since(start)\n\tif el > 400*time.Millisecond {\n\t\tt.Fatalf("overran time: %v", el)\n\t}\n\tt.Logf("200ms: depth %d, nodes %d, %.0f knps", r.Depth, r.Nodes, float64(r.Nodes)/r.Elapsed.Seconds()/1000)\n}\n\n// Приёмка: глубина >= 6 за секунду и 0 аллокаций в поиске.\nfunc TestDepthInOneSecond(t *testing.T) {\n\ts := New(20)\n\tps := positions(200, 5)\n\tvar minD, sumD, cnt int = 99, 0, 0\n\tfor i := 0; i < len(ps); i += 20 { // 10 разных позиций\n\t\ts.Clear()\n\t\tr := s.Search(ps[i], Limits{Time: time.Second})\n\t\tt.Logf("pos %3d: depth %2d nodes %7d %.0f knps", i, r.Depth, r.Nodes, float64(r.Nodes)/r.Elapsed.Seconds()/1000)\n\t\tif i == 0 && r.Depth < 6 { // позиция 0 = старт партии\n\t\t\tt.Errorf("start position: depth %d < 6 in 1s", r.Depth)\n\t\t}\n\t\tif r.Depth < minD {\n\t\t\tminD = r.Depth\n\t\t}\n\t\tsumD += r.Depth\n\t\tcnt++\n\t}\n\tt.Logf("min depth %d, avg %.1f", minD, float64(sumD)/float64(cnt))\n}\n\nfunc TestNoAllocs(t *testing.T) {\n\ts := New(18)\n\tb := positions(50, 6)[30]\n\ts.Search(b, Limits{MaxDepth: 4}) // прогрев\n\tif a := testing.AllocsPerRun(5, func() { s.Search(b, Limits{MaxDepth: 5}) }); a != 0 {\n\t\tt.Fatalf("allocs per search = %v, want 0", a)\n\t}\n}\n\nfunc BenchmarkSearchDepth5(bm *testing.B) {\n\ts := New(20)\n\tps := positions(100, 7)\n\tbm.ResetTimer()\n\tfor i := 0; i < bm.N; i++ {\n\t\ts.Clear()\n\t\ts.Search(ps[(i*7)%len(ps)], Limits{MaxDepth: 5})\n\t}\n}\n\nfunc TestExtensionNoAllocsAndLegal(t *testing.T) {\n\ts := New(18)\n\ts.ExtendWalls = true\n\tps := positions(40, 9)\n\tfor i, b := range ps {\n\t\tr := s.Search(b, Limits{MaxDepth: 3})\n\t\tif !inMoves(b.Moves(nil), r.Move) {\n\t\t\tt.Fatalf("pos %d: illegal move with extensions", i)\n\t\t}\n\t}\n\tb := ps[30]\n\ts.Search(b, Limits{MaxDepth: 4})\n\tif a := testing.AllocsPerRun(5, func() { s.Search(b, Limits{MaxDepth: 4}) }); a != 0 {\n\t\tt.Fatalf("allocs with extensions = %v", a)\n\t}\n}\n\n')

os.makedirs('match', exist_ok=True)
with open('match/match.go', 'w', encoding='utf-8') as f:
    f.write('// Package match: арена бот-против-бота. Игры парные: одна и та же случайная\n// дебютная расстановка играется дважды со сменой цвета, чтобы убрать шум дебюта.\npackage match\n\nimport (\n\t"math/rand"\n\t"sync"\n\t"time"\n\n\t"quoridor/board"\n\t"quoridor/movegen"\n\t"quoridor/path"\n\t"quoridor/search"\n\t"quoridor/threat"\n)\n\ntype Bot interface {\n\tChoose(b board.Board) board.Move\n}\n\n// Walker: «иди по кратчайшему пути», стен не ставит. Базовая планка силы.\ntype Walker struct{}\n\nfunc (Walker) Choose(b board.Board) board.Move {\n\tme := int(b.Turn)\n\td := path.Field(&b, board.GoalRow(me))\n\tvar buf [8]uint8\n\tbest, bd := uint8(0), int8(127)\n\tfor _, to := range b.PawnMoves(me, buf[:0]) {\n\t\tif d[to] < bd {\n\t\t\tbest, bd = to, d[to]\n\t\t}\n\t}\n\treturn board.PawnMove(int(best))\n}\n\n// Searcher: бот на поиске с фиксированной глубиной и/или временем.\ntype Searcher struct {\n\tS    *search.Searcher\n\tLim  search.Limits\n\tLast search.Result // итог последнего поиска (оценка, глубина, узлы) — для интерфейсов и отладки\n}\n\nfunc NewSearcher(ttBits uint, lim search.Limits) *Searcher {\n\treturn &Searcher{S: search.New(ttBits), Lim: lim}\n}\n\nfunc (s *Searcher) Choose(b board.Board) board.Move {\n\ts.Last = s.S.Search(b, s.Lim)\n\treturn s.Last.Move\n}\n\ntype Record struct {\n\tMoves  []board.Move\n\tWinner int // 0, 1 или -1 (ничья по лимиту ходов)\n\tASide  int // какой стороной играл бот A (0/1), -1 если неизвестно\n}\n\n// PlayGame играет партию: p0 ходит первым. opening играется заранее без участия ботов.\nfunc PlayGame(p0, p1 Bot, opening []board.Move, maxPlies int) Record {\n\tb := board.New()\n\trec := Record{Winner: -1, ASide: -1}\n\tfor _, m := range opening {\n\t\tb = b.Apply(m)\n\t\trec.Moves = append(rec.Moves, m)\n\t}\n\tbots := [2]Bot{p0, p1}\n\tfor len(rec.Moves) < maxPlies {\n\t\tif w := b.Winner(); w >= 0 {\n\t\t\trec.Winner = w\n\t\t\treturn rec\n\t\t}\n\t\tm := bots[b.Turn].Choose(b)\n\t\tb = b.Apply(m)\n\t\trec.Moves = append(rec.Moves, m)\n\t}\n\tif w := b.Winner(); w >= 0 {\n\t\trec.Winner = w\n\t}\n\treturn rec\n}\n\n// RandomOpening: n случайных легальных полуходов.\nfunc RandomOpening(rng *rand.Rand, n int) []board.Move {\n\tb := board.New()\n\tvar out []board.Move\n\tfor i := 0; i < n; i++ {\n\t\tms := movegen.Gen(&b, nil)\n\t\tm := ms[rng.Intn(len(ms))]\n\t\tout = append(out, m)\n\t\tb = b.Apply(m)\n\t}\n\treturn out\n}\n\ntype Score struct {\n\tAWins, BWins, Draws int\n\tGames               []Record // с точки зрения порядка (Aпервый, Aвторой) по парам\n\tElapsed             time.Duration\n}\n\nfunc (s Score) Total() int { return s.AWins + s.BWins + s.Draws }\n\n// Rate: доля очков A (ничья = 0.5).\nfunc (s Score) Rate() float64 {\n\tif s.Total() == 0 {\n\t\treturn 0\n\t}\n\treturn (float64(s.AWins) + 0.5*float64(s.Draws)) / float64(s.Total())\n}\n\n// Run играет pairs пар партий (2*pairs партий). newA/newB создают свежих ботов на каждую партию\n// (у ботов есть состояние вроде TT, а игры идут параллельно).\nfunc Run(newA, newB func() Bot, pairs, openingPlies, maxPlies, workers int, seed int64) Score {\n\tstart := time.Now()\n\ttype job struct{ pair int }\n\tjobs := make(chan int)\n\tvar mu sync.Mutex\n\tvar sc Score\n\tvar wg sync.WaitGroup\n\tfor w := 0; w < workers; w++ {\n\t\twg.Add(1)\n\t\tgo func() {\n\t\t\tdefer wg.Done()\n\t\t\tfor i := range jobs {\n\t\t\t\trng := rand.New(rand.NewSource(seed + int64(i)))\n\t\t\t\top := RandomOpening(rng, openingPlies)\n\t\t\t\t// A играет первой, затем B играет первой с тем же дебютом\n\t\t\t\tr1 := PlayGame(newA(), newB(), op, maxPlies)\n\t\t\t\tr2 := PlayGame(newB(), newA(), op, maxPlies)\n\t\t\t\tr1.ASide, r2.ASide = 0, 1\n\t\t\t\tmu.Lock()\n\t\t\t\tfor k, r := range []Record{r1, r2} {\n\t\t\t\t\taSide := k // в партии 0 A — игрок 0, в партии 1 A — игрок 1\n\t\t\t\t\tswitch {\n\t\t\t\t\tcase r.Winner < 0:\n\t\t\t\t\t\tsc.Draws++\n\t\t\t\t\tcase r.Winner == aSide:\n\t\t\t\t\t\tsc.AWins++\n\t\t\t\t\tdefault:\n\t\t\t\t\t\tsc.BWins++\n\t\t\t\t\t}\n\t\t\t\t\tsc.Games = append(sc.Games, r)\n\t\t\t\t}\n\t\t\t\tmu.Unlock()\n\t\t\t}\n\t\t}()\n\t}\n\tfor i := 0; i < pairs; i++ {\n\t\tjobs <- i\n\t}\n\tclose(jobs)\n\twg.Wait()\n\tsc.Elapsed = time.Since(start)\n\treturn sc\n}\n\n// ThreatBot: перед обычным поиском ищет форсированную ловушку стенами (threat.Find) и играет её первую стену.\ntype ThreatBot struct {\n\tBase  Bot\n\tDepth int // моих стен в ловушке (обычно 2)\n\tThr   int // требуемый выигрыш пути сверх ходьбы\n}\n\nfunc (t *ThreatBot) Choose(b board.Board) board.Move {\n\tif b.Walls[b.Turn] > 0 {\n\t\tif r := threat.Find(&b, t.Depth, t.Thr); r.Found {\n\t\t\treturn r.Move\n\t\t}\n\t}\n\treturn t.Base.Choose(b)\n}\n\n')

os.makedirs('match', exist_ok=True)
with open('match/match_test.go', 'w', encoding='utf-8') as f:
    f.write('package match\n\nimport (\n\t"runtime"\n\t"testing"\n\n\t"quoridor/board"\n\t"quoridor/eval"\n\t"quoridor/search"\n)\n\nfunc workers() int {\n\tw := runtime.GOMAXPROCS(0)\n\tif w < 1 {\n\t\tw = 1\n\t}\n\treturn w\n}\n\n// Приёмка этапа 5: бот с eval v1 уверенно обыгрывает Walker.\nfunc TestV1BeatsWalker(t *testing.T) {\n\tpairs := 60\n\tif testing.Short() {\n\t\tpairs = 8\n\t}\n\tnewA := func() Bot { return NewSearcher(18, search.Limits{MaxDepth: 4}) }\n\tnewB := func() Bot { return Walker{} }\n\tsc := Run(newA, newB, pairs, 2, 300, workers(), 1) // дебют: 2 случайных полухода\n\tt.Logf("search(d4) vs walker: +%d -%d =%d  rate %.3f  (%v)", sc.AWins, sc.BWins, sc.Draws, sc.Rate(), sc.Elapsed)\n\tif sc.Rate() < 0.9 {\n\t\tt.Errorf("expected >= 90%% points, got %.1f%%", 100*sc.Rate())\n\t}\n}\n\n// Контроль арены: Walker против Walker — выигрывает тот, кто ходит первым.\nfunc TestWalkerMirror(t *testing.T) {\n\tsc := Run(func() Bot { return Walker{} }, func() Bot { return Walker{} }, 10, 4, 300, workers(), 2)\n\tt.Logf("walker vs walker: +%d -%d =%d", sc.AWins, sc.BWins, sc.Draws)\n\tif sc.Draws != 0 {\n\t\tt.Errorf("walkers must always finish, draws=%d", sc.Draws)\n\t}\n}\n\nfunc v2bot(depth int) Bot {\n\ts := NewSearcher(18, search.Limits{MaxDepth: depth})\n\tp := eval.DefaultV2()\n\ts.S.Eval = p.Eval\n\treturn s\n}\n\nfunc v1bot(depth int) Bot {\n\ts := NewSearcher(18, search.Limits{MaxDepth: depth})\n\ts.S.Eval = func(b board.Board) int { return eval.V1(&b) }\n\treturn s\n}\n\n// Приёмка этапа 6: v2 выигрывает у v1 со статистической значимостью (SPRT: H0 = 0 Elo, H1 = +30 Elo, alpha=beta=5%).\nfunc TestV2BeatsV1(t *testing.T) {\n\tif testing.Short() {\n\t\tt.Skip("долгий матч")\n\t}\n\tr := RunSPRT(func() Bot { return v2bot(3) }, func() Bot { return v1bot(3) },\n\t\tSPRT{Elo0: 0, Elo1: 30, Alpha: 0.05, Beta: 0.05}, 300, 50, 4, 300, workers(), 61)\n\tt.Logf("v2 vs v1 (depth 3): %s (%v)", r, r.Score.Elapsed.Round(1e8))\n\tif r.Verdict != "H1" {\n\t\tt.Errorf("v2 does not beat v1 significantly: %s", r)\n\t}\n}\n\nfunc TestSPRTMath(t *testing.T) {\n\ts := SPRT{Elo0: 0, Elo1: 30, Alpha: 0.05, Beta: 0.05}\n\tif l := s.LLR(300, 0, 200); l <= 0 {\n\t\tt.Errorf("better score must give positive LLR, got %v", l)\n\t}\n\tif l := s.LLR(200, 0, 300); l >= 0 {\n\t\tt.Errorf("worse score must give negative LLR, got %v", l)\n\t}\n\tlo, hi := s.Bounds()\n\tif lo >= 0 || hi <= 0 {\n\t\tt.Errorf("bounds %v %v", lo, hi)\n\t}\n\te, a, b := EloCI(55, 0, 45)\n\tif !(a < e && e < b) || e < 30 || e > 40 {\n\t\tt.Errorf("EloCI(55,0,45) = %.1f [%.1f %.1f]", e, a, b)\n\t}\n}\n\n')

os.makedirs('match', exist_ok=True)
with open('match/sprt.go', 'w', encoding='utf-8') as f:
    f.write('package match\n\nimport (\n\t"fmt"\n\t"math"\n)\n\n// SPRT: последовательный тест отношения правдоподобия по Elo (H0: elo0, H1: elo1).\n// Трёхисходная (W/D/L) нормальная аппроксимация LLR, как в fishtest.\ntype SPRT struct {\n\tElo0, Elo1  float64\n\tAlpha, Beta float64\n}\n\nfunc eloToScore(elo float64) float64 { return 1 / (1 + math.Pow(10, -elo/400)) }\n\nfunc scoreToElo(s float64) float64 {\n\tif s <= 0 {\n\t\treturn math.Inf(-1)\n\t}\n\tif s >= 1 {\n\t\treturn math.Inf(1)\n\t}\n\treturn -400 * math.Log10(1/s-1)\n}\n\n// Bounds: LLR <= lo принимает H0, LLR >= hi принимает H1.\nfunc (t SPRT) Bounds() (lo, hi float64) {\n\treturn math.Log(t.Beta / (1 - t.Alpha)), math.Log((1 - t.Beta) / t.Alpha)\n}\n\nfunc (t SPRT) LLR(w, d, l int) float64 {\n\tn := float64(w + d + l)\n\tif n == 0 {\n\t\treturn 0\n\t}\n\ts := (float64(w) + 0.5*float64(d)) / n\n\tv := (float64(w)*(1-s)*(1-s) + float64(d)*(0.5-s)*(0.5-s) + float64(l)*s*s) / n\n\tif v < 1e-6 {\n\t\tv = 1e-6\n\t}\n\ts0, s1 := eloToScore(t.Elo0), eloToScore(t.Elo1)\n\treturn n * (s1 - s0) * (2*s - s0 - s1) / (2 * v)\n}\n\n// EloCI: Elo и 95%-доверительный интервал по W/D/L.\nfunc EloCI(w, d, l int) (elo, lo, hi float64) {\n\tn := float64(w + d + l)\n\tif n == 0 {\n\t\treturn 0, math.Inf(-1), math.Inf(1)\n\t}\n\ts := (float64(w) + 0.5*float64(d)) / n\n\tv := (float64(w)*(1-s)*(1-s) + float64(d)*(0.5-s)*(0.5-s) + float64(l)*s*s) / n\n\tse := math.Sqrt(v / n)\n\treturn scoreToElo(s), scoreToElo(s - 1.96*se), scoreToElo(s + 1.96*se)\n}\n\ntype SPRTResult struct {\n\tScore   Score\n\tLLR     float64\n\tVerdict string // "H1" (A сильнее), "H0" (не сильнее), "undecided"\n}\n\nfunc (r SPRTResult) String() string {\n\te, lo, hi := EloCI(r.Score.AWins, r.Score.Draws, r.Score.BWins)\n\treturn fmt.Sprintf("+%d -%d =%d  Elo %.0f [%.0f, %.0f]  LLR %.2f  verdict %s",\n\t\tr.Score.AWins, r.Score.BWins, r.Score.Draws, e, lo, hi, r.LLR, r.Verdict)\n}\n\n// RunSPRT играет партиями-парами пачками по batch, пока LLR не выйдет за границы или не кончатся maxPairs.\nfunc RunSPRT(newA, newB func() Bot, t SPRT, maxPairs, batch, openingPlies, maxPlies, workers int, seed int64) SPRTResult {\n\tlo, hi := t.Bounds()\n\tvar total Score\n\tres := SPRTResult{Verdict: "undecided"}\n\tfor done := 0; done < maxPairs; done += batch {\n\t\tn := batch\n\t\tif done+n > maxPairs {\n\t\t\tn = maxPairs - done\n\t\t}\n\t\tsc := Run(newA, newB, n, openingPlies, maxPlies, workers, seed+int64(done)*7919)\n\t\ttotal.AWins += sc.AWins\n\t\ttotal.BWins += sc.BWins\n\t\ttotal.Draws += sc.Draws\n\t\ttotal.Elapsed += sc.Elapsed\n\t\tres.LLR = t.LLR(total.AWins, total.Draws, total.BWins)\n\t\tif res.LLR >= hi {\n\t\t\tres.Verdict = "H1"\n\t\t\tbreak\n\t\t}\n\t\tif res.LLR <= lo {\n\t\t\tres.Verdict = "H0"\n\t\t\tbreak\n\t\t}\n\t}\n\tres.Score = total\n\treturn res\n}\n\n')

os.makedirs('match', exist_ok=True)
with open('match/record.go', 'w', encoding='utf-8') as f:
    f.write('package match\n\nimport (\n\t"bufio"\n\t"fmt"\n\t"io"\n\t"strconv"\n\t"strings"\n\n\t"quoridor/board"\n)\n\n// ---- запись ходов и позиций в текст ----\n\n// FormatMove: пешка "p<клетка 0..80>", горизонтальная стена "h<r><c>", вертикальная "v<r><c>".\nfunc FormatMove(m board.Move) string {\n\tswitch m.Kind() {\n\tcase board.KindPawn:\n\t\treturn "p" + strconv.Itoa(m.Idx())\n\tcase board.KindH:\n\t\treturn fmt.Sprintf("h%d%d", m.Idx()/8, m.Idx()%8)\n\tdefault:\n\t\treturn fmt.Sprintf("v%d%d", m.Idx()/8, m.Idx()%8)\n\t}\n}\n\nfunc ParseMove(s string) (board.Move, error) {\n\tif len(s) < 2 {\n\t\treturn 0, fmt.Errorf("bad move %q", s)\n\t}\n\tswitch s[0] {\n\tcase \'p\':\n\t\tn, err := strconv.Atoi(s[1:])\n\t\tif err != nil || n < 0 || n > 80 {\n\t\t\treturn 0, fmt.Errorf("bad pawn move %q", s)\n\t\t}\n\t\treturn board.PawnMove(n), nil\n\tcase \'h\', \'v\':\n\t\tif len(s) != 3 || s[1] < \'0\' || s[1] > \'7\' || s[2] < \'0\' || s[2] > \'7\' {\n\t\t\treturn 0, fmt.Errorf("bad wall move %q", s)\n\t\t}\n\t\treturn board.WallMove(s[0] == \'h\', int(s[1]-\'0\'), int(s[2]-\'0\')), nil\n\t}\n\treturn 0, fmt.Errorf("bad move %q", s)\n}\n\n// String: "A=<сторона A> W=<победитель> ходы...". Партия воспроизводится с board.New().\nfunc (r Record) String() string {\n\tvar sb strings.Builder\n\tfmt.Fprintf(&sb, "A=%d W=%d", r.ASide, r.Winner)\n\tfor _, m := range r.Moves {\n\t\tsb.WriteByte(\' \')\n\t\tsb.WriteString(FormatMove(m))\n\t}\n\treturn sb.String()\n}\n\nfunc ParseRecord(line string) (Record, error) {\n\tf := strings.Fields(line)\n\tif len(f) < 2 || !strings.HasPrefix(f[0], "A=") || !strings.HasPrefix(f[1], "W=") {\n\t\treturn Record{}, fmt.Errorf("bad record header: %q", line)\n\t}\n\ta, err1 := strconv.Atoi(f[0][2:])\n\tw, err2 := strconv.Atoi(f[1][2:])\n\tif err1 != nil || err2 != nil {\n\t\treturn Record{}, fmt.Errorf("bad record header: %q", line)\n\t}\n\tr := Record{ASide: a, Winner: w}\n\tfor _, s := range f[2:] {\n\t\tm, err := ParseMove(s)\n\t\tif err != nil {\n\t\t\treturn Record{}, err\n\t\t}\n\t\tr.Moves = append(r.Moves, m)\n\t}\n\treturn r, nil\n}\n\nfunc WriteGames(w io.Writer, gs []Record) error {\n\tbw := bufio.NewWriter(w)\n\tfor _, g := range gs {\n\t\tif _, err := bw.WriteString(g.String() + "\\n"); err != nil {\n\t\t\treturn err\n\t\t}\n\t}\n\treturn bw.Flush()\n}\n\nfunc ReadGames(r io.Reader) ([]Record, error) {\n\tvar out []Record\n\tsc := bufio.NewScanner(r)\n\tsc.Buffer(make([]byte, 1<<20), 1<<20)\n\tfor sc.Scan() {\n\t\tline := strings.TrimSpace(sc.Text())\n\t\tif line == "" || strings.HasPrefix(line, "#") {\n\t\t\tcontinue\n\t\t}\n\t\tg, err := ParseRecord(line)\n\t\tif err != nil {\n\t\t\treturn nil, err\n\t\t}\n\t\tout = append(out, g)\n\t}\n\treturn out, sc.Err()\n}\n\n// Replay: позиции ПЕРЕД каждым ходом партии (len == len(Moves)); проверяет заодно легальность ходов.\nfunc Replay(r Record) ([]board.Board, error) {\n\tb := board.New()\n\tvar out []board.Board\n\tfor i, m := range r.Moves {\n\t\tout = append(out, b)\n\t\tif m.Kind() == board.KindPawn {\n\t\t\tok := false\n\t\t\tvar buf [8]uint8\n\t\t\tfor _, to := range b.PawnMoves(int(b.Turn), buf[:0]) {\n\t\t\t\tif int(to) == m.Idx() {\n\t\t\t\t\tok = true\n\t\t\t\t}\n\t\t\t}\n\t\t\tif !ok {\n\t\t\t\treturn nil, fmt.Errorf("ply %d: illegal pawn move %s", i, FormatMove(m))\n\t\t\t}\n\t\t} else if _, ok := b.WithWall(m.Kind() == board.KindH, m.Idx()/8, m.Idx()%8); !ok {\n\t\t\treturn nil, fmt.Errorf("ply %d: illegal wall %s", i, FormatMove(m))\n\t\t}\n\t\tb = b.Apply(m)\n\t}\n\treturn out, nil\n}\n\nfunc FormatBoard(b board.Board) string {\n\treturn fmt.Sprintf("HW=%#x VW=%#x P=%d,%d W=%d,%d T=%d", b.HW, b.VW, b.P[0], b.P[1], b.Walls[0], b.Walls[1], b.Turn)\n}\n\nfunc ParseBoard(s string) (board.Board, error) {\n\tvar b board.Board\n\tvar hw, vw uint64\n\tvar p0, p1, w0, w1, t int\n\tif _, err := fmt.Sscanf(strings.TrimSpace(s), "HW=%v VW=%v P=%d,%d W=%d,%d T=%d", &hw, &vw, &p0, &p1, &w0, &w1, &t); err != nil {\n\t\treturn b, fmt.Errorf("bad board %q: %v", s, err)\n\t}\n\tif p0 < 0 || p0 > 80 || p1 < 0 || p1 > 80 || w0 < 0 || w0 > 10 || w1 < 0 || w1 > 10 || (t != 0 && t != 1) {\n\t\treturn b, fmt.Errorf("board out of range: %q", s)\n\t}\n\tb.HW, b.VW = hw, vw\n\tb.P = [2]uint8{uint8(p0), uint8(p1)}\n\tb.Walls = [2]uint8{uint8(w0), uint8(w1)}\n\tb.Turn = uint8(t)\n\treturn b, nil\n}\n\n// ---- дамп проигранных позиций в регрессионный набор ----\n\ntype Position struct {\n\tBoard  board.Board\n\tPlayed board.Move // что сыграл проигравший в этой позиции\n\tGame   int\n\tPly    int\n}\n\n// LossPositions: для партий, которые проиграл бот who (0 = A, 1 = B), берёт его последние lastN\n// позиций перед ходом (с ходом, который он сделал). Это материал для разбора и регрессионных тестов.\nfunc LossPositions(gs []Record, who, lastN int) []Position {\n\tvar out []Position\n\tfor gi, g := range gs {\n\t\tif g.Winner < 0 || g.ASide < 0 {\n\t\t\tcontinue\n\t\t}\n\t\tside := g.ASide\n\t\tif who == 1 {\n\t\t\tside = 1 - g.ASide\n\t\t}\n\t\tif g.Winner == side { // этот бот выиграл\n\t\t\tcontinue\n\t\t}\n\t\tps, err := Replay(g)\n\t\tif err != nil {\n\t\t\tcontinue\n\t\t}\n\t\tvar mine []Position\n\t\tfor i, b := range ps {\n\t\t\tif int(b.Turn) == side {\n\t\t\t\tmine = append(mine, Position{Board: b, Played: g.Moves[i], Game: gi, Ply: i})\n\t\t\t}\n\t\t}\n\t\tif len(mine) > lastN {\n\t\t\tmine = mine[len(mine)-lastN:]\n\t\t}\n\t\tout = append(out, mine...)\n\t}\n\treturn out\n}\n\nfunc WritePositions(w io.Writer, ps []Position) error {\n\tbw := bufio.NewWriter(w)\n\tfor _, p := range ps {\n\t\tif _, err := fmt.Fprintf(bw, "%s | %s | game %d ply %d\\n", FormatBoard(p.Board), FormatMove(p.Played), p.Game, p.Ply); err != nil {\n\t\t\treturn err\n\t\t}\n\t}\n\treturn bw.Flush()\n}\n\nfunc ReadPositions(r io.Reader) ([]Position, error) {\n\tvar out []Position\n\tsc := bufio.NewScanner(r)\n\tfor sc.Scan() {\n\t\tline := strings.TrimSpace(sc.Text())\n\t\tif line == "" || strings.HasPrefix(line, "#") {\n\t\t\tcontinue\n\t\t}\n\t\tparts := strings.Split(line, "|")\n\t\tif len(parts) < 2 {\n\t\t\treturn nil, fmt.Errorf("bad position line %q", line)\n\t\t}\n\t\tb, err := ParseBoard(parts[0])\n\t\tif err != nil {\n\t\t\treturn nil, err\n\t\t}\n\t\tm, err := ParseMove(strings.TrimSpace(parts[1]))\n\t\tif err != nil {\n\t\t\treturn nil, err\n\t\t}\n\t\tp := Position{Board: b, Played: m}\n\t\tif len(parts) > 2 {\n\t\t\tfmt.Sscanf(strings.TrimSpace(parts[2]), "game %d ply %d", &p.Game, &p.Ply)\n\t\t}\n\t\tout = append(out, p)\n\t}\n\treturn out, sc.Err()\n}\n\n')

os.makedirs('match', exist_ok=True)
with open('match/record_test.go', 'w', encoding='utf-8') as f:
    f.write('package match\n\nimport (\n\t"bytes"\n\t"math/rand"\n\t"strings"\n\t"testing"\n\n\t"quoridor/board"\n\t"quoridor/search"\n)\n\nfunc randomRecords(n int) []Record {\n\tsc := Run(func() Bot { return NewSearcher(14, search.Limits{MaxDepth: 2}) }, func() Bot { return Walker{} }, n/2, 4, 300, 2, 5)\n\treturn sc.Games\n}\n\nfunc TestRecordRoundTrip(t *testing.T) {\n\tgs := randomRecords(20)\n\tvar buf bytes.Buffer\n\tif err := WriteGames(&buf, gs); err != nil {\n\t\tt.Fatal(err)\n\t}\n\tback, err := ReadGames(&buf)\n\tif err != nil || len(back) != len(gs) {\n\t\tt.Fatalf("read: %v (%d vs %d)", err, len(back), len(gs))\n\t}\n\tfor i := range gs {\n\t\tif gs[i].String() != back[i].String() || gs[i].ASide < 0 {\n\t\t\tt.Fatalf("game %d differs:\\n%s\\n%s", i, gs[i], back[i])\n\t\t}\n\t\tif _, err := Replay(back[i]); err != nil {\n\t\t\tt.Fatalf("game %d replay: %v", i, err)\n\t\t}\n\t}\n}\n\nfunc TestBoardRoundTrip(t *testing.T) {\n\trng := rand.New(rand.NewSource(1))\n\tfor _, g := range randomRecords(10) {\n\t\tps, _ := Replay(g)\n\t\tfor _, b := range ps {\n\t\t\tgot, err := ParseBoard(FormatBoard(b))\n\t\t\tif err != nil || got != b {\n\t\t\t\tt.Fatalf("round trip failed: %v\\n%+v\\n%+v", err, b, got)\n\t\t\t}\n\t\t}\n\t\t_ = rng\n\t}\n\tif _, err := ParseBoard("garbage"); err == nil {\n\t\tt.Fatal("garbage must fail")\n\t}\n}\n\nfunc TestParseMoveErrors(t *testing.T) {\n\tfor _, s := range []string{"", "x1", "p99", "h99", "h8", "p-1"} {\n\t\tif _, err := ParseMove(s); err == nil {\n\t\t\tt.Errorf("%q must fail", s)\n\t\t}\n\t}\n\tfor _, m := range []board.Move{board.PawnMove(0), board.PawnMove(80), board.WallMove(true, 7, 7), board.WallMove(false, 0, 3)} {\n\t\tback, err := ParseMove(FormatMove(m))\n\t\tif err != nil || back != m {\n\t\t\tt.Errorf("move %v -> %q -> %v (%v)", m, FormatMove(m), back, err)\n\t\t}\n\t}\n}\n\nfunc TestLossPositions(t *testing.T) {\n\t// A = поиск, B = Walker; A-поражения встречаются редко, но B проигрывает часто\n\tgs := randomRecords(40)\n\tlostB := LossPositions(gs, 1, 3)\n\tif len(lostB) == 0 {\n\t\tt.Fatal("expected some positions where walker lost")\n\t}\n\tfor _, p := range lostB {\n\t\tg := gs[p.Game]\n\t\tif g.Winner != g.ASide || int(p.Board.Turn) != 1-g.ASide { // B проиграла: A победила, ход в позиции за B\n\t\t\tt.Fatalf("wrong position selection: %+v game %v", p, g.String())\n\t\t}\n\t}\n\tvar buf bytes.Buffer\n\tif err := WritePositions(&buf, lostB); err != nil {\n\t\tt.Fatal(err)\n\t}\n\tback, err := ReadPositions(strings.NewReader(buf.String()))\n\tif err != nil || len(back) != len(lostB) {\n\t\tt.Fatalf("read positions: %v", err)\n\t}\n\tfor i := range back {\n\t\tif back[i].Board != lostB[i].Board || back[i].Played != lostB[i].Played {\n\t\t\tt.Fatalf("position %d differs", i)\n\t\t}\n\t}\n}\n\nfunc TestBotFactory(t *testing.T) {\n\tfor _, ok := range []string{"walker", "v1", "v2", "v2:Flex=5,Walls=150"} {\n\t\tif _, err := BotFactory(ok, 2, 0, 14); err != nil {\n\t\t\tt.Errorf("%q: %v", ok, err)\n\t\t}\n\t}\n\tfor _, bad := range []string{"", "v3", "v2:Flex", "v2:Nope=1", "v2:Flex=x"} {\n\t\tif _, err := BotFactory(bad, 2, 0, 14); err == nil {\n\t\t\tt.Errorf("%q must fail", bad)\n\t\t}\n\t}\n\tf, _ := BotFactory("v2:Walls=200", 2, 0, 14)\n\tb := board.New()\n\tif m := f().Choose(b); m == 0 && false {\n\t\tt.Fatal(m)\n\t}\n}\n\n')

os.makedirs('match', exist_ok=True)
with open('match/spec.go', 'w', encoding='utf-8') as f:
    f.write('package match\n\nimport (\n\t"fmt"\n\t"strings"\n\t"time"\n\n\t"quoridor/board"\n\t"quoridor/eval"\n\t"quoridor/search"\n)\n\n// BotFactory строит фабрику ботов по описанию:\n//\n//\twalker                    — всегда идёт по кратчайшему пути\n//\tv1                        — поиск + eval.V1\n//\tv2                        — поиск + eval.V2 с DefaultV2\n//\tv2:Flex=5,Walls=150       — то же с переопределёнными весами\n//\n// depth > 0 — фиксированная глубина, ms > 0 — время на ход.\nfunc BotFactory(spec string, depth, ms int, ttBits uint) (func() Bot, error) {\n\tname, args, _ := strings.Cut(spec, ":")\n\tlim := search.Limits{MaxDepth: depth}\n\tif ms > 0 {\n\t\tlim.Time = time.Duration(ms) * time.Millisecond\n\t}\n\tswitch name {\n\tcase "walker":\n\t\treturn func() Bot { return Walker{} }, nil\n\tcase "v1":\n\t\treturn func() Bot {\n\t\t\ts := NewSearcher(ttBits, lim)\n\t\t\ts.S.Eval = func(b board.Board) int { return eval.V1(&b) }\n\t\t\treturn s\n\t\t}, nil\n\tcase "v2":\n\t\tp, err := eval.ParseParams(args)\n\t\tif err != nil {\n\t\t\treturn nil, err\n\t\t}\n\t\treturn func() Bot {\n\t\t\ts := NewSearcher(ttBits, lim)\n\t\t\tpp := p\n\t\t\ts.S.Eval = pp.Eval\n\t\t\treturn s\n\t\t}, nil\n\t}\n\treturn nil, fmt.Errorf("unknown bot %q (walker | v1 | v2[:Name=Value,...])", spec)\n}\n\n')

os.makedirs('threat', exist_ok=True)
with open('threat/threat.go', 'w', encoding='utf-8') as f:
    f.write('// Package threat: узкий поиск «только стены» для форсированных ловушек.\n//\n// Ищем последовательность: моя стена, ЛЮБОЙ ответ соперника, моя стена, ... (depth моих стен),\n// после которой разность путей (путь соперника минус мой) больше, чем при простой ходьбе, на thr и более.\n// Это И/ИЛИ-поиск: по моим стенам — «существует», по ответам соперника — «для всех».\n//\n// Классика: «одна стена отдельно (почти не удлиняет путь), вторая запирает». У соперника нет\n// единственного ответа, который снимает обе угрозы сразу.\n//\n// Мерка выигрыша: gain = (dOpp - dMe) в конце - (dOpp - dMe) в корне - 1.\n// Ходьба даёт «ничью» по темпу (я +1 шаг, он +1 шаг), поэтому «+1» вычитается.\n//\n// Полнота: для depth <= 2 ответы соперника перебираются ВСЕ (найденная ловушка действительно\n// форсированная при тех кандидатах, что перебрали мы). Для depth 3 ответы урезаны эвристикой.\npackage threat\n\nimport (\n\t"quoridor/board"\n\t"quoridor/movegen"\n\t"quoridor/path"\n)\n\ntype Result struct {\n\tFound bool\n\tMove  board.Move // первая стена\n\tGain  int        // достигнутый gain (>= thr)\n\tNodes int\n}\n\ntype finder struct {\n\tthr        int\n\texhaustive bool // test-only: все легальные стены на всех шагах\n\tme         int\n\td0         int\n\tnodes      int\n}\n\nvar dr = [4]int{-1, 1, 0, 0}\nvar dc = [4]int{0, 0, -1, 1}\n\nfunc margin(b *board.Board, me int) int { return path.Len(b, 1-me) - path.Len(b, me) }\n\n// Find: depth = число моих стен в последовательности (1..3).\nfunc Find(b *board.Board, depth, thr int) Result {\n\tf := finder{thr: thr, me: int(b.Turn)}\n\treturn f.run(b, depth)\n}\n\n// FindExhaustive: эталон для тестов (медленнее, кандидаты — все легальные стены).\nfunc FindExhaustive(b *board.Board, depth, thr int) Result {\n\tf := finder{thr: thr, me: int(b.Turn), exhaustive: true}\n\treturn f.run(b, depth)\n}\n\nfunc (f *finder) run(b *board.Board, depth int) Result {\n\tif b.Winner() >= 0 || b.Walls[b.Turn] == 0 || depth < 1 {\n\t\treturn Result{}\n\t}\n\tf.d0 = margin(b, f.me)\n\tres := Result{}\n\tfor _, w := range f.wallCands(b, depth == 1) {\n\t\tnb := b.Apply(w)\n\t\tf.nodes++\n\t\tok := false\n\t\tgain := 0\n\t\tif depth == 1 {\n\t\t\tgain = margin(&nb, f.me) - f.d0 - 1\n\t\t\tok = gain >= f.thr\n\t\t} else {\n\t\t\tok = f.and(&nb, depth-1)\n\t\t\tif ok {\n\t\t\t\tgain = f.thr // нижняя оценка: gain >= thr\n\t\t\t}\n\t\t}\n\t\tif ok {\n\t\t\tres = Result{Found: true, Move: w, Gain: gain}\n\t\t\tbreak\n\t\t}\n\t}\n\tres.Nodes = f.nodes\n\treturn res\n}\n\n// and: ход соперника; ловушка верна, если после КАЖДОГО его ответа у меня есть продолжение.\nfunc (f *finder) and(b *board.Board, k int) bool {\n\tfor _, r := range f.replies(b, k) {\n\t\tnb := b.Apply(r)\n\t\tf.nodes++\n\t\tif nb.Winner() >= 0 {\n\t\t\treturn false\n\t\t}\n\t\tif !f.or(&nb, k) {\n\t\t\treturn false\n\t\t}\n\t}\n\treturn true\n}\n\n// or: мой ход, нужно k стен подряд (с ответами между ними).\nfunc (f *finder) or(b *board.Board, k int) bool {\n\tif b.Walls[b.Turn] == 0 {\n\t\treturn false\n\t}\n\tfor _, w := range f.wallCands(b, k == 1) {\n\t\tnb := b.Apply(w)\n\t\tf.nodes++\n\t\tif k == 1 {\n\t\t\tif margin(&nb, f.me)-f.d0-1 >= f.thr {\n\t\t\t\treturn true\n\t\t\t}\n\t\t} else if f.and(&nb, k-1) {\n\t\t\treturn true\n\t\t}\n\t}\n\treturn false\n}\n\n// replies: ответы соперника: все ходы пешкой и стены (полный перебор при k<=1, т.е. для depth<=2).\nfunc (f *finder) replies(b *board.Board, k int) []board.Move {\n\tif k <= 1 || f.exhaustive {\n\t\treturn movegen.GenAll(b, nil)\n\t}\n\treturn movegen.Gen(b, nil) // для глубины 3: только отсечённые ответы (эвристика)\n}\n\n// wallCands: мои стены-кандидаты. final=true: последняя стена в цепочке (должна давать выигрыш пути сама),\n// иначе «подготовительная»: стены вблизи кратчайших путей соперника (в том числе почти без эффекта).\nfunc (f *finder) wallCands(b *board.Board, final bool) []board.Move {\n\tif f.exhaustive {\n\t\tvar out []board.Move\n\t\tfor _, m := range movegen.GenAll(b, nil) {\n\t\t\tif m.Kind() != board.KindPawn {\n\t\t\t\tout = append(out, m)\n\t\t\t}\n\t\t}\n\t\treturn out\n\t}\n\tif final {\n\t\tvar out []board.Move\n\t\tfor _, m := range movegen.Gen(b, nil) {\n\t\t\tif m.Kind() != board.KindPawn {\n\t\t\t\tout = append(out, m)\n\t\t\t}\n\t\t}\n\t\treturn out\n\t}\n\treturn f.prepCands(b)\n}\n\n// prepCands: легальные стены, касающиеся клеток в радиусе 2 от кратчайших путей соперника.\nfunc (f *finder) prepCands(b *board.Board) []board.Move {\n\topp := 1 - int(b.Turn)\n\td := path.Field(b, board.GoalRow(opp))\n\tstart := int(b.P[opp])\n\tvar onDag, mark [81]bool\n\tvar q [81]uint8\n\tq[0], onDag[start] = uint8(start), true\n\th, t := 0, 1\n\tfor h < t {\n\t\tcur := int(q[h])\n\t\th++\n\t\tr, c := cur/9, cur%9\n\t\tif d[cur] <= 0 {\n\t\t\tcontinue\n\t\t}\n\t\tfor k := 0; k < 4; k++ {\n\t\t\tif !b.CanStep(r, c, k) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tn := (r+dr[k])*9 + c + dc[k]\n\t\t\tif d[n] == d[cur]-1 && !onDag[n] {\n\t\t\t\tonDag[n] = true\n\t\t\t\tq[t] = uint8(n)\n\t\t\t\tt++\n\t\t\t}\n\t\t}\n\t}\n\tconst rad = 2\n\tfor i := 0; i < 81; i++ {\n\t\tif !onDag[i] {\n\t\t\tcontinue\n\t\t}\n\t\tr, c := i/9, i%9\n\t\tfor rr := r - rad; rr <= r+rad; rr++ {\n\t\t\tfor cc := c - rad; cc <= c+rad; cc++ {\n\t\t\t\tif rr >= 0 && rr < 9 && cc >= 0 && cc < 9 {\n\t\t\t\t\tmark[rr*9+cc] = true\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t}\n\tvar out []board.Move\n\tfor r := 0; r < 8; r++ {\n\t\tfor c := 0; c < 8; c++ {\n\t\t\tif !(mark[r*9+c] || mark[r*9+c+1] || mark[(r+1)*9+c] || mark[(r+1)*9+c+1]) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tif movegen.LegalWall(b, true, r, c) {\n\t\t\t\tout = append(out, board.WallMove(true, r, c))\n\t\t\t}\n\t\t\tif movegen.LegalWall(b, false, r, c) {\n\t\t\t\tout = append(out, board.WallMove(false, r, c))\n\t\t\t}\n\t\t}\n\t}\n\treturn out\n}\n\n')

os.makedirs('threat', exist_ok=True)
with open('threat/threat_test.go', 'w', encoding='utf-8') as f:
    f.write('package threat\n\nimport (\n\t"fmt"\n\t"math/rand"\n\t"testing"\n\t"time"\n\n\t"quoridor/board"\n\t"quoridor/movegen"\n)\n\n// Позиции: случайные партии с повышенной долей стен (в них ловушки встречаются чаще).\nfunc positions(n int, seed int64) []board.Board {\n\trng := rand.New(rand.NewSource(seed))\n\tvar out []board.Board\n\tfor len(out) < n {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 70 && len(out) < n && b.Winner() < 0; ply++ {\n\t\t\tif ply >= 8 && b.Walls[b.Turn] > 0 {\n\t\t\t\tout = append(out, b)\n\t\t\t}\n\t\t\tms := movegen.Gen(&b, nil)\n\t\t\t// 60% ходов — стены (если есть), чтобы поле быстрее заполнялось\n\t\t\tvar pick board.Move\n\t\t\tvar walls []board.Move\n\t\t\tfor _, m := range ms {\n\t\t\t\tif m.Kind() != board.KindPawn {\n\t\t\t\t\twalls = append(walls, m)\n\t\t\t\t}\n\t\t\t}\n\t\t\tif len(walls) > 0 && rng.Intn(100) < 60 {\n\t\t\t\tpick = walls[rng.Intn(len(walls))]\n\t\t\t} else {\n\t\t\t\tpick = ms[rng.Intn(len(ms))]\n\t\t\t}\n\t\t\tb = b.Apply(pick)\n\t\t}\n\t}\n\treturn out\n}\n\nfunc TestFindSoundness(t *testing.T) { // найденный ход всегда легален\n\tfor i, b := range positions(300, 1) {\n\t\tr := Find(&b, 2, 2)\n\t\tif r.Found {\n\t\t\tif !movegen.LegalWall(&b, r.Move.Kind() == board.KindH, r.Move.Idx()/8, r.Move.Idx()%8) {\n\t\t\t\tt.Fatalf("pos %d: illegal trap move", i)\n\t\t\t}\n\t\t}\n\t}\n}\n\nfunc TestDepth1MatchesBruteForce(t *testing.T) { // depth 1 — это просто «стена с gain >= thr»\n\tfor i, b := range positions(300, 2) {\n\t\ta, e := Find(&b, 1, 2), FindExhaustive(&b, 1, 2)\n\t\tif a.Found != e.Found {\n\t\t\tt.Fatalf("pos %d: pruned=%v exhaustive=%v\\n%+v", i, a.Found, e.Found, b)\n\t\t}\n\t}\n}\n\n// Сводка по ловушкам глубины 2: сколько позиций с настоящей двухстеночной ловушкой\n// (одна стена не достаёт thr), и какую часть находит быстрый Find.\nfunc TestTrapRecall(t *testing.T) {\n\tn := 1500\n\tif testing.Short() {\n\t\tn = 300\n\t}\n\tpos := positions(n, 3)\n\ttrap, found, one, fast, slow := 0, 0, 0, time.Duration(0), time.Duration(0)\n\tvar regress []board.Board\n\tfor _, b := range pos {\n\t\tif FindExhaustive(&b, 1, 2).Found {\n\t\t\tone++\n\t\t\tcontinue\n\t\t}\n\t\tt0 := time.Now()\n\t\te := FindExhaustive(&b, 2, 2)\n\t\tslow += time.Since(t0)\n\t\tif !e.Found {\n\t\t\tcontinue\n\t\t}\n\t\ttrap++\n\t\tt0 = time.Now()\n\t\ta := Find(&b, 2, 2)\n\t\tfast += time.Since(t0)\n\t\tif a.Found {\n\t\t\tfound++\n\t\t} else if len(regress) < 5 {\n\t\t\tregress = append(regress, b)\n\t\t}\n\t}\n\tt.Logf("positions %d: single-wall wins %d, true 2-wall traps %d, found by Find %d", len(pos), one, trap, found)\n\tif trap > 0 {\n\t\tt.Logf("recall %.0f%%; time per trap-position: exhaustive %v, Find %v", 100*float64(found)/float64(trap), slow/time.Duration(trap), fast/time.Duration(trap))\n\t\tif float64(found) < 0.9*float64(trap) {\n\t\t\tfor _, b := range regress {\n\t\t\t\tfmt.Printf("missed: %+v\\n", b)\n\t\t\t}\n\t\t\tt.Errorf("recall too low: %d/%d", found, trap)\n\t\t}\n\t}\n}\n\n')

os.makedirs('threat', exist_ok=True)
with open('threat/regression_test.go', 'w', encoding='utf-8') as f:
    f.write('package threat\n\nimport (\n\t"testing"\n\t"time"\n\n\t"quoridor/board"\n\t"quoridor/path"\n)\n\n// Регрессионные позиции: двухстеночные ловушки, где ни одна стена сама не даёт gain >= 2,\n// но после первой стены у соперника нет ответа, который снимает угрозу.\n// (Позиции получены полным перебором FindExhaustive и зафиксированы здесь.)\nvar traps = []board.Board{\n\t{HW: 0x800140010100024, VW: 0x0, P: [2]uint8{4, 67}, Walls: [2]uint8{6, 7}, Turn: 0},\n\t{HW: 0x800140010101024, VW: 0x400000000000000, P: [2]uint8{4, 67}, Walls: [2]uint8{5, 6}, Turn: 0},\n\t{HW: 0xa08282000000000, VW: 0x1000000000000008, P: [2]uint8{4, 76}, Walls: [2]uint8{6, 6}, Turn: 0},\n\t{HW: 0xa08292000000000, VW: 0x1000000000000008, P: [2]uint8{4, 76}, Walls: [2]uint8{5, 6}, Turn: 1},\n\t{HW: 0xa09292008000000, VW: 0x1000000000000008, P: [2]uint8{4, 76}, Walls: [2]uint8{4, 5}, Turn: 1},\n\t{HW: 0x40400000140000, VW: 0x1800001000000012, P: [2]uint8{4, 76}, Walls: [2]uint8{5, 6}, Turn: 1},\n\t{HW: 0x40400000540000, VW: 0x1800001200000012, P: [2]uint8{4, 76}, Walls: [2]uint8{4, 5}, Turn: 1},\n\t{HW: 0x800104405000890, VW: 0x200000000000, P: [2]uint8{4, 76}, Walls: [2]uint8{5, 5}, Turn: 0},\n}\n\n// Независимая проверка ловушки: полный перебор ходов через board.Moves (медленный эталон правил).\nfunc verifyTrap(t *testing.T, b board.Board, w board.Move, thr int) {\n\tt.Helper()\n\tme := int(b.Turn)\n\td0 := path.Len(&b, 1-me) - path.Len(&b, me)\n\tnb := b.Apply(w)\n\tfor _, r := range nb.Moves(nil) {\n\t\tn2 := nb.Apply(r)\n\t\tif n2.Winner() >= 0 {\n\t\t\tt.Fatalf("opponent wins at once after trap wall, reply %v", r)\n\t\t}\n\t\tok := false\n\t\tfor _, w2 := range n2.Moves(nil) {\n\t\t\tif w2.Kind() == board.KindPawn {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tn3 := n2.Apply(w2)\n\t\t\tif path.Len(&n3, 1-me)-path.Len(&n3, me)-d0-1 >= thr {\n\t\t\t\tok = true\n\t\t\t\tbreak\n\t\t\t}\n\t\t}\n\t\tif !ok {\n\t\t\tt.Fatalf("reply %v refutes the trap wall %v", r, w)\n\t\t}\n\t}\n}\n\nfunc TestRegressionTraps(t *testing.T) {\n\tfor i, b := range traps {\n\t\tif one := Find(&b, 1, 2); one.Found {\n\t\t\tt.Fatalf("trap %d: single wall already wins (not a pure 2-wall trap)", i)\n\t\t}\n\t\tr := Find(&b, 2, 2)\n\t\tif !r.Found {\n\t\t\tt.Fatalf("trap %d: Find missed the trap", i)\n\t\t}\n\t\tverifyTrap(t, b, r.Move, 2)\n\t}\n}\n\n// Найденные на случайных позициях ловушки независимо подтверждаются board.Moves.\nfunc TestFoundTrapsVerified(t *testing.T) {\n\tn, checked := 400, 0\n\tif testing.Short() {\n\t\tn = 100\n\t}\n\tfor _, b := range positions(n, 4) {\n\t\tr := Find(&b, 2, 2)\n\t\tif r.Found {\n\t\t\tverifyTrap(t, b, r.Move, 2)\n\t\t\tchecked++\n\t\t}\n\t}\n\tt.Logf("independently verified %d traps", checked)\n}\n\nfunc TestTiming(t *testing.T) {\n\tps := positions(200, 5)\n\tdepths := []int{1, 2, 3}\n\tif testing.Short() {\n\t\tdepths, ps = []int{1, 2}, ps[:50]\n\t}\n\tfor _, d := range depths {\n\t\tt0 := time.Now()\n\t\tfound := 0\n\t\tfor _, b := range ps {\n\t\t\tif Find(&b, d, 2).Found {\n\t\t\t\tfound++\n\t\t\t}\n\t\t}\n\t\tel := time.Since(t0)\n\t\tt.Logf("depth %d: %d/%d positions with a forced win-of-path, %v per position", d, found, len(ps), el/time.Duration(len(ps)))\n\t}\n}\n\n')

os.makedirs('tune', exist_ok=True)
with open('tune/spsa.go', 'w', encoding='utf-8') as f:
    f.write('// Package tune: настройка весов оценки методом SPSA через матчи «theta+ против theta-».\npackage tune\n\nimport (\n\t"math"\n\t"math/rand"\n\n\t"quoridor/eval"\n\t"quoridor/match"\n\t"quoridor/search"\n)\n\n// Spec: один настраиваемый параметр.\ntype Spec struct {\n\tName         string\n\tInit, Lo, Hi float64\n\tC            float64 // размах возмущения (в единицах параметра)\n}\n\ntype SPSA struct {\n\tSpecs []Spec\n\tTheta []float64\n\tLR    float64 // начальный шаг (в долях C за итерацию)\n\tIters int     // ожидаемое число итераций (шаг убывает к ~LR/4)\n\tK     int\n\trng   *rand.Rand\n}\n\nfunc New(specs []Spec, lr float64, iters int, seed int64) *SPSA {\n\ts := &SPSA{Specs: specs, LR: lr, Iters: iters, rng: rand.New(rand.NewSource(seed))}\n\tfor _, sp := range specs {\n\t\ts.Theta = append(s.Theta, sp.Init)\n\t}\n\treturn s\n}\n\nfunc (s *SPSA) clamp(i int, v float64) float64 {\n\treturn math.Max(s.Specs[i].Lo, math.Min(s.Specs[i].Hi, v))\n}\n\n// Step делает одну итерацию. score(plus, minus) возвращает долю очков plus против minus в [0,1].\nfunc (s *SPSA) Step(score func(plus, minus []float64) float64) (sc float64) {\n\tn := len(s.Theta)\n\tdelta := make([]float64, n)\n\tplus := make([]float64, n)\n\tminus := make([]float64, n)\n\tck := math.Pow(1+float64(s.K)/float64(s.Iters), -0.101) // слабое затухание возмущения\n\tfor i := range s.Theta {\n\t\tdelta[i] = 1\n\t\tif s.rng.Intn(2) == 0 {\n\t\t\tdelta[i] = -1\n\t\t}\n\t\tc := s.Specs[i].C * ck\n\t\tplus[i] = s.clamp(i, s.Theta[i]+c*delta[i])\n\t\tminus[i] = s.clamp(i, s.Theta[i]-c*delta[i])\n\t}\n\tsc = score(plus, minus)\n\t// Шаг: двигаем theta в сторону победителя матча. (plus-minus)/2 = c*delta, поэтому\n\t// theta += lr * c*delta * (2*score-1): при ничьей (score=0.5) шага нет, при полной победе plus\n\t// сдвиг на lr*c в сторону plus. lr затухает с ростом итерации.\n\tlr := s.LR / math.Pow(1+3*float64(s.K)/float64(s.Iters), 0.6)\n\tfor i := range s.Theta {\n\t\ts.Theta[i] = s.clamp(i, s.Theta[i]+lr*(plus[i]-minus[i])/2*(2*sc-1)*2)\n\t}\n\ts.K++\n\treturn sc\n}\n\n// ---- привязка к eval.Params ----\n\nfunc DefaultSpecs() []Spec {\n\td := eval.DefaultV2()\n\treturn []Spec{\n\t\t{"Tempo", float64(d.Tempo), 0, 100, 10},\n\t\t{"Flex", float64(d.Flex), 0, 20, 3},\n\t\t{"Cut", float64(d.Cut), 0, 40, 5},\n\t\t{"Exit", float64(d.Exit), 0, 80, 10},\n\t\t{"Walls", float64(d.Walls), 60, 350, 25},\n\t\t{"WallsNarrow", float64(d.WallsNarrow), 0, 20, 3},\n\t}\n}\n\nfunc ToParams(specs []Spec, theta []float64) eval.Params {\n\tp := eval.DefaultV2()\n\tfor i, sp := range specs {\n\t\tp.Set(sp.Name, int(math.Round(theta[i])))\n\t}\n\treturn p\n}\n\n// MatchScore: функция оценки для SPSA — матч depth-фиксированных ботов plus против minus.\nfunc MatchScore(specs []Spec, depth, pairs, openingPlies, workers int, seed func() int64) func(plus, minus []float64) float64 {\n\treturn func(plus, minus []float64) float64 {\n\t\tpp, pm := ToParams(specs, plus), ToParams(specs, minus)\n\t\tmk := func(p eval.Params) func() match.Bot {\n\t\t\treturn func() match.Bot {\n\t\t\t\tb := match.NewSearcher(16, search.Limits{MaxDepth: depth})\n\t\t\t\tq := p\n\t\t\t\tb.S.Eval = q.Eval\n\t\t\t\treturn b\n\t\t\t}\n\t\t}\n\t\tsc := match.Run(mk(pp), mk(pm), pairs, openingPlies, 300, workers, seed())\n\t\treturn sc.Rate()\n\t}\n}\n\n')

os.makedirs('tune', exist_ok=True)
with open('tune/spsa_test.go', 'w', encoding='utf-8') as f:
    f.write('package tune\n\nimport (\n\t"math"\n\t"math/rand"\n\t"testing"\n\n\t"quoridor/eval"\n)\n\n// Синтетический «матч»: исход определяется тем, чей theta ближе к скрытому оптимуму, плюс шум.\nfunc TestSPSAConverges(t *testing.T) {\n\topt := []float64{40, 8, 150}\n\tspecs := []Spec{{"a", 0, -100, 100, 8}, {"b", 20, 0, 30, 2}, {"c", 300, 0, 400, 30}}\n\trng := rand.New(rand.NewSource(1))\n\tdist := func(x []float64) float64 {\n\t\td := 0.0\n\t\tfor i := range x {\n\t\t\tz := (x[i] - opt[i]) / specs[i].C\n\t\t\td += z * z\n\t\t}\n\t\treturn d\n\t}\n\ts := New(specs, 1.0, 600, 2)\n\tfor k := 0; k < 600; k++ {\n\t\ts.Step(func(plus, minus []float64) float64 {\n\t\t\tp := 1 / (1 + math.Exp(0.15*(dist(plus)-dist(minus)))) // ближе к оптимуму -> чаще побеждает\n\t\t\t// матч из 80 партий: биномиальный шум\n\t\t\tw := 0\n\t\t\tfor g := 0; g < 80; g++ {\n\t\t\t\tif rng.Float64() < p {\n\t\t\t\t\tw++\n\t\t\t\t}\n\t\t\t}\n\t\t\treturn float64(w) / 80\n\t\t})\n\t}\n\tfor i := range opt {\n\t\tif math.Abs(s.Theta[i]-opt[i]) > 2.5*specs[i].C {\n\t\t\tt.Errorf("param %s = %.1f, optimum %.1f (C=%.1f)", specs[i].Name, s.Theta[i], opt[i], specs[i].C)\n\t\t}\n\t}\n\tt.Logf("theta = %.1f (optimum %v)", s.Theta, opt)\n}\n\nfunc TestToParams(t *testing.T) {\n\tsp := DefaultSpecs()\n\tth := make([]float64, len(sp))\n\tfor i := range sp {\n\t\tth[i] = sp[i].Init\n\t}\n\tp := ToParams(sp, th)\n\tif p != eval.DefaultV2() {\n\t\tt.Fatalf("init theta must reproduce DefaultV2: %+v", p)\n\t}\n\tth[4] = 1e9 // за границей — зажимается только в SPSA, ToParams просто округляет\n\t_ = ToParams(sp, th)\n}\n\n')

os.makedirs('movegen', exist_ok=True)
with open('movegen/movegen.go', 'w', encoding='utf-8') as f:
    f.write('// Package movegen: отсечённый генератор ходов с ленивой проверкой легальности стен.\n//\n// Идея:\n//  1. Кандидаты-стены: те, что перекрывают ребро кратчайших путей соперника\n//     (от его пешки к цели) или любое ребро у его пешки.\n//  2. Правило контактов: стена может отрезать путь, только если хотя бы 2 из её\n//     3 точек (концы и середина) касаются границы доски или уже стоящих стен.\n//     При <2 контактов легальность (наличие пути) гарантирована без BFS.\n//  3. Ленивость: Candidates отдаёт ходы без проверки путей, Legal доделывает её\n//     только для стен с >=2 контактов; Gen = Candidates + Legal.\npackage movegen\n\nimport (\n\t"math/bits"\n\n\t"quoridor/board"\n\t"quoridor/path"\n)\n\nvar dr = [4]int{-1, 1, 0, 0}\nvar dc = [4]int{0, 0, -1, 1}\n\nfunc has(m uint64, r, c int) bool {\n\tif r < 0 || r > 7 || c < 0 || c > 7 {\n\t\treturn false\n\t}\n\treturn m&(1<<uint(r*8+c)) != 0\n}\n\n// Занятые точки решётки 10x10: бит j в occ[i] = точка на пересечении линий (i,j).\ntype occ [10]uint16\n\nfunc mark(o *occ, i, j int) { o[i] |= 1 << uint(j) }\n\n// pointsOf: концы и середина стены в координатах решётки.\nfunc pointsOf(h bool, r, c int) [3][2]int {\n\tif h {\n\t\treturn [3][2]int{{r + 1, c}, {r + 1, c + 1}, {r + 1, c + 2}}\n\t}\n\treturn [3][2]int{{r, c + 1}, {r + 1, c + 1}, {r + 2, c + 1}}\n}\n\nfunc occupied(b *board.Board) (o occ) {\n\tfor m := b.HW; m != 0; m &= m - 1 {\n\t\ti := tz(m)\n\t\tfor _, p := range pointsOf(true, i/8, i%8) {\n\t\t\tmark(&o, p[0], p[1])\n\t\t}\n\t}\n\tfor m := b.VW; m != 0; m &= m - 1 {\n\t\ti := tz(m)\n\t\tfor _, p := range pointsOf(false, i/8, i%8) {\n\t\t\tmark(&o, p[0], p[1])\n\t\t}\n\t}\n\treturn\n}\n\nfunc tz(m uint64) int { return bits.TrailingZeros64(m) }\n\nfunc contacts(o *occ, h bool, r, c int) (n int) {\n\tfor _, p := range pointsOf(h, r, c) {\n\t\tif p[0] == 0 || p[0] == 9 || p[1] == 0 || p[1] == 9 || o[p[0]]>>uint(p[1])&1 != 0 {\n\t\t\tn++\n\t\t}\n\t}\n\treturn\n}\n\n// placeable: все проверки, кроме существования пути (границы, запас, перекрытие, крест).\nfunc placeable(b *board.Board, h bool, r, c int) bool {\n\tif r < 0 || r > 7 || c < 0 || c > 7 || b.Walls[b.Turn] == 0 {\n\t\treturn false\n\t}\n\tif h {\n\t\treturn !(has(b.HW, r, c) || has(b.HW, r, c-1) || has(b.HW, r, c+1) || has(b.VW, r, c))\n\t}\n\treturn !(has(b.VW, r, c) || has(b.VW, r-1, c) || has(b.VW, r+1, c) || has(b.HW, r, c))\n}\n\n// pathsOK: после установки стены оба игрока ещё имеют путь.\nfunc pathsOK(b *board.Board, h bool, r, c int) bool {\n\tnb := *b\n\tif h {\n\t\tnb.HW |= 1 << uint(r*8+c)\n\t} else {\n\t\tnb.VW |= 1 << uint(r*8+c)\n\t}\n\treturn path.Len(&nb, 0) >= 0 && path.Len(&nb, 1) >= 0\n}\n\n// LegalWall: то же, что board.WithWall(...).ok, но с правилом контактов.\nfunc LegalWall(b *board.Board, h bool, r, c int) bool {\n\tif !placeable(b, h, r, c) {\n\t\treturn false\n\t}\n\to := occupied(b)\n\tif contacts(&o, h, r, c) < 2 {\n\t\treturn true\n\t}\n\treturn pathsOK(b, h, r, c)\n}\n\n// Legal: ленивая проверка хода из Candidates (пешечные ходы всегда легальны).\nfunc Legal(b *board.Board, m board.Move) bool {\n\tif m.Kind() == board.KindPawn {\n\t\treturn true\n\t}\n\th := m.Kind() == board.KindH\n\tr, c := m.Idx()/8, m.Idx()%8\n\to := occupied(b)\n\tif contacts(&o, h, r, c) < 2 {\n\t\treturn true\n\t}\n\treturn pathsOK(b, h, r, c)\n}\n\n// blockers помечает стены, перекрывающие шаг из (r,c) в направлении k.\nfunc blockers(k, r, c int, hc, vc *uint64) {\n\tset := func(m *uint64, rr, cc int) {\n\t\tif rr >= 0 && rr <= 7 && cc >= 0 && cc <= 7 {\n\t\t\t*m |= 1 << uint(rr*8+cc)\n\t\t}\n\t}\n\tswitch k {\n\tcase board.Up:\n\t\tset(hc, r-1, c)\n\t\tset(hc, r-1, c-1)\n\tcase board.Down:\n\t\tset(hc, r, c)\n\t\tset(hc, r, c-1)\n\tcase board.Left:\n\t\tset(vc, r, c-1)\n\t\tset(vc, r-1, c-1)\n\tdefault:\n\t\tset(vc, r, c)\n\t\tset(vc, r-1, c)\n\t}\n}\n\n// wallCandidates: маски стен, которые стоит рассматривать (без проверки легальности).\nfunc wallCandidates(b *board.Board) (hc, vc uint64) {\n\topp := 1 - int(b.Turn)\n\td := path.Field(b, board.GoalRow(opp))\n\tstart := int(b.P[opp])\n\tvar seen [81]bool\n\tvar q [81]uint8\n\tq[0], seen[start] = uint8(start), true\n\th, t := 0, 1\n\tfor h < t {\n\t\tcur := int(q[h])\n\t\th++\n\t\tr, c := cur/9, cur%9\n\t\tfor k := 0; k < 4; k++ {\n\t\t\tif !b.CanStep(r, c, k) {\n\t\t\t\tcontinue\n\t\t\t}\n\t\t\tn := (r+dr[k])*9 + c + dc[k]\n\t\t\t// ребро на кратчайшем пути соперника, либо любое ребро у его пешки\n\t\t\tif (d[cur] > 0 && d[n] == d[cur]-1) || cur == start {\n\t\t\t\tblockers(k, r, c, &hc, &vc)\n\t\t\t}\n\t\t\tif d[cur] > 0 && d[n] == d[cur]-1 && !seen[n] {\n\t\t\t\tseen[n] = true\n\t\t\t\tq[t] = uint8(n)\n\t\t\t\tt++\n\t\t\t}\n\t\t}\n\t}\n\treturn\n}\n\n// Candidates: ходы пешки + отсечённые стены, которые прошли дешёвые проверки\n// (границы, запас, перекрытие, крест). Пути у стен с >=2 контактами НЕ проверены:\n// перед применением вызывай Legal.\nfunc Candidates(b *board.Board, out []board.Move) []board.Move {\n\tif b.Winner() >= 0 {\n\t\treturn out\n\t}\n\tvar buf [8]uint8\n\tfor _, to := range b.PawnMoves(int(b.Turn), buf[:0]) {\n\t\tout = append(out, board.PawnMove(int(to)))\n\t}\n\tif b.Walls[b.Turn] == 0 {\n\t\treturn out\n\t}\n\thc, vc := wallCandidates(b)\n\tfor m := hc; m != 0; m &= m - 1 {\n\t\ti := tz(m)\n\t\tif placeable(b, true, i/8, i%8) {\n\t\t\tout = append(out, board.WallMove(true, i/8, i%8))\n\t\t}\n\t}\n\tfor m := vc; m != 0; m &= m - 1 {\n\t\ti := tz(m)\n\t\tif placeable(b, false, i/8, i%8) {\n\t\t\tout = append(out, board.WallMove(false, i/8, i%8))\n\t\t}\n\t}\n\treturn out\n}\n\n// Gen: полностью легальные отсечённые ходы.\nfunc Gen(b *board.Board, out []board.Move) []board.Move {\n\tstart := len(out)\n\tout = Candidates(b, out)\n\tw := start\n\tfor _, m := range out[start:] {\n\t\tif Legal(b, m) {\n\t\t\tout[w] = m\n\t\t\tw++\n\t\t}\n\t}\n\treturn out[:w]\n}\n\n// GenAll: ВСЕ легальные ходы (без отсечения кандидатов), но быстро: стены через LegalWall\n// (правило контактов), поэтому в десятки раз быстрее board.Moves. Результат совпадает с board.Moves.\nfunc GenAll(b *board.Board, out []board.Move) []board.Move {\n\tif b.Winner() >= 0 {\n\t\treturn out\n\t}\n\tvar buf [8]uint8\n\tfor _, to := range b.PawnMoves(int(b.Turn), buf[:0]) {\n\t\tout = append(out, board.PawnMove(int(to)))\n\t}\n\tif b.Walls[b.Turn] == 0 {\n\t\treturn out\n\t}\n\tfor r := 0; r < 8; r++ {\n\t\tfor c := 0; c < 8; c++ {\n\t\t\tif LegalWall(b, true, r, c) {\n\t\t\t\tout = append(out, board.WallMove(true, r, c))\n\t\t\t}\n\t\t\tif LegalWall(b, false, r, c) {\n\t\t\t\tout = append(out, board.WallMove(false, r, c))\n\t\t\t}\n\t\t}\n\t}\n\treturn out\n}\n\n')

os.makedirs('movegen', exist_ok=True)
with open('movegen/movegen_test.go', 'w', encoding='utf-8') as f:
    f.write('package movegen\n\nimport (\n\t"math/rand"\n\t"testing"\n\n\t"quoridor/board"\n\t"quoridor/path"\n)\n\n// ---- позиции ----\n\n// Случайные легальные партии.\nfunc randomPositions(n int, seed int64) []board.Board {\n\trng := rand.New(rand.NewSource(seed))\n\tvar out []board.Board\n\tfor len(out) < n {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 100 && len(out) < n && b.Winner() < 0; ply++ {\n\t\t\tout = append(out, b)\n\t\t\tif rng.Intn(100) < 45 && b.Walls[b.Turn] > 0 {\n\t\t\t\tmoved := false\n\t\t\t\tfor try := 0; try < 30 && !moved; try++ {\n\t\t\t\t\th := rng.Intn(2) == 0\n\t\t\t\t\tr, c := rng.Intn(8), rng.Intn(8)\n\t\t\t\t\tif _, ok := b.WithWall(h, r, c); ok {\n\t\t\t\t\t\tb = b.Apply(board.WallMove(h, r, c))\n\t\t\t\t\t\tmoved = true\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\tif moved {\n\t\t\t\t\tcontinue\n\t\t\t\t}\n\t\t\t}\n\t\t\tms := b.PawnMoves(int(b.Turn), make([]uint8, 0, 8))\n\t\t\tb = b.Apply(board.PawnMove(int(ms[rng.Intn(len(ms))])))\n\t\t}\n\t}\n\treturn out\n}\n\n// score1: разность путей после хода с точки зрения походившего (больше = лучше).\nfunc score1(b board.Board, m board.Move) int {\n\tme := int(b.Turn)\n\tnb := b.Apply(m)\n\tif w := nb.Winner(); w >= 0 {\n\t\tif w == me {\n\t\t\treturn 100\n\t\t}\n\t\treturn -100\n\t}\n\treturn path.Len(&nb, 1-me) - path.Len(&nb, me)\n}\n\n// Партии «жадных» ботов (1 шаг по score1 из отсечённого генератора, 15% случайных ходов):\n// позиции ближе к реальной игре, чем чисто случайные.\nfunc greedyPositions(n int, seed int64) []board.Board {\n\trng := rand.New(rand.NewSource(seed))\n\tvar out []board.Board\n\tfor len(out) < n {\n\t\tb := board.New()\n\t\tfor ply := 0; ply < 120 && len(out) < n && b.Winner() < 0; ply++ {\n\t\t\tout = append(out, b)\n\t\t\tms := Gen(&b, nil)\n\t\t\tvar m board.Move\n\t\t\tif rng.Intn(100) < 15 {\n\t\t\t\tm = ms[rng.Intn(len(ms))]\n\t\t\t} else {\n\t\t\t\tbest, bs := ms[0], -1000\n\t\t\t\tfor _, x := range ms {\n\t\t\t\t\ts := score1(b, x)*4 + rng.Intn(4) // случайный разбор ничьих\n\t\t\t\t\tif s > bs {\n\t\t\t\t\t\tbest, bs = x, s\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t\tm = best\n\t\t\t}\n\t\t\tb = b.Apply(m)\n\t\t}\n\t}\n\treturn out\n}\n\nfunc inSet(ms []board.Move, m board.Move) bool {\n\tfor _, x := range ms {\n\t\tif x == m {\n\t\t\treturn true\n\t\t}\n\t}\n\treturn false\n}\n\n// ---- тесты ----\n\n// Правило контактов не должно менять легальность ни одной стены.\nfunc TestLegalWallMatchesWithWall(t *testing.T) {\n\tps := append(randomPositions(1500, 1), greedyPositions(1500, 2)...)\n\tskipped, total := 0, 0\n\tfor i, b := range ps {\n\t\to := occupied(&b)\n\t\tfor _, h := range []bool{true, false} {\n\t\t\tfor r := 0; r < 8; r++ {\n\t\t\t\tfor c := 0; c < 8; c++ {\n\t\t\t\t\t_, want := b.WithWall(h, r, c)\n\t\t\t\t\tif got := LegalWall(&b, h, r, c); got != want {\n\t\t\t\t\t\tt.Fatalf("pos %d wall h=%v (%d,%d): got %v want %v\\n%+v", i, h, r, c, got, want, b)\n\t\t\t\t\t}\n\t\t\t\t\tif placeable(&b, h, r, c) {\n\t\t\t\t\t\ttotal++\n\t\t\t\t\t\tif contacts(&o, h, r, c) < 2 {\n\t\t\t\t\t\t\tskipped++\n\t\t\t\t\t\t}\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\t}\n\t}\n\tt.Logf("BFS skipped for %d of %d placeable walls (%.0f%%)", skipped, total, 100*float64(skipped)/float64(total))\n}\n\nfunc TestGenSubsetOfFullAndHasPawnMoves(t *testing.T) {\n\tfor i, b := range append(randomPositions(500, 3), greedyPositions(500, 4)...) {\n\t\tfull := b.Moves(nil)\n\t\tgen := Gen(&b, nil)\n\t\tfor _, m := range gen {\n\t\t\tif !inSet(full, m) {\n\t\t\t\tt.Fatalf("pos %d: Gen produced illegal move %v\\n%+v", i, m, b)\n\t\t\t}\n\t\t}\n\t\tfor _, m := range full {\n\t\t\tif m.Kind() == board.KindPawn && !inSet(gen, m) {\n\t\t\t\tt.Fatalf("pos %d: pawn move %v missing", i, m)\n\t\t\t}\n\t\t}\n\t}\n}\n\nfunc TestBranching(t *testing.T) {\n\tfor name, ps := range map[string][]board.Board{\n\t\t"random": randomPositions(3000, 5),\n\t\t"greedy": greedyPositions(3000, 6),\n\t} {\n\t\tsum, sumFull, max := 0, 0, 0\n\t\tfor _, b := range ps {\n\t\t\tn := len(Gen(&b, nil))\n\t\t\tsum += n\n\t\t\tsumFull += len(b.Moves(nil))\n\t\t\tif n > max {\n\t\t\t\tmax = n\n\t\t\t}\n\t\t}\n\t\tavg := float64(sum) / float64(len(ps))\n\t\tt.Logf("%s: avg branching %.1f (full %.1f), max %d", name, avg, float64(sumFull)/float64(len(ps)), max)\n\t\tif avg > 25 {\n\t\t\tt.Errorf("%s: average branching %.1f > 25", name, avg)\n\t\t}\n\t}\n}\n\n// value2: минимакс на 2 хода (мой ход, ответ соперника) по разности путей.\nfunc value2(b board.Board, gen func(*board.Board, []board.Move) []board.Move) int {\n\tme := int(b.Turn)\n\tbest := -1000\n\tfor _, m := range gen(&b, nil) {\n\t\tnb := b.Apply(m)\n\t\tif nb.Winner() >= 0 {\n\t\t\tif nb.Winner() == me {\n\t\t\t\treturn 100\n\t\t\t}\n\t\t\tcontinue\n\t\t}\n\t\tworst := 1000\n\t\tfor _, r := range gen(&nb, nil) {\n\t\t\tnn := nb.Apply(r)\n\t\t\tvar v int\n\t\t\tif w := nn.Winner(); w >= 0 {\n\t\t\t\tv = -100 // раз ответ соперника привёл к победе, это его победа\n\t\t\t} else {\n\t\t\t\tv = path.Len(&nn, 1-me) - path.Len(&nn, me)\n\t\t\t}\n\t\t\tif v < worst {\n\t\t\t\tworst = v\n\t\t\t}\n\t\t}\n\t\tif worst > best {\n\t\t\tbest = worst\n\t\t}\n\t}\n\treturn best\n}\n\nfunc bestValue1(b board.Board, ms []board.Move) int {\n\tbest := -1000\n\tfor _, m := range ms {\n\t\tif s := score1(b, m); s > best {\n\t\t\tbest = s\n\t\t}\n\t}\n\treturn best\n}\n\n// Лучшая достижимая оценка (1 и 2 хода) полным и отсечённым генератором должна совпадать.\nfunc TestSameBestValue(t *testing.T) {\n\tn := 300\n\tif testing.Short() {\n\t\tn = 40\n\t}\n\tps := append(randomPositions(n/2, 7), greedyPositions(n/2, 8)...)\n\tsame1, same2 := 0, 0\n\tfor _, b := range ps {\n\t\tif bestValue1(b, b.Moves(nil)) == bestValue1(b, Gen(&b, nil)) {\n\t\t\tsame1++\n\t\t}\n\t\tfull := func(x *board.Board, out []board.Move) []board.Move { return x.Moves(out) }\n\t\tif value2(b, full) == value2(b, Gen) {\n\t\t\tsame2++\n\t\t}\n\t}\n\tt.Logf("same best value: 1-ply %d/%d, 2-ply %d/%d", same1, len(ps), same2, len(ps))\n\tif float64(same1) < 0.95*float64(len(ps)) || float64(same2) < 0.85*float64(len(ps)) {\n\t\tt.Errorf("pruning loses too much")\n\t}\n}\n\nfunc BenchmarkGen(bm *testing.B) {\n\tps := greedyPositions(500, 9)\n\tbuf := make([]board.Move, 0, 160)\n\tbm.ResetTimer()\n\tfor i := 0; i < bm.N; i++ {\n\t\tbuf = Gen(&ps[i%len(ps)], buf[:0])\n\t}\n}\n\nfunc BenchmarkFullMoves(bm *testing.B) {\n\tps := greedyPositions(500, 9)\n\tbuf := make([]board.Move, 0, 160)\n\tbm.ResetTimer()\n\tfor i := 0; i < bm.N; i++ {\n\t\tbuf = ps[i%len(ps)].Moves(buf[:0])\n\t}\n}\n\n')

os.makedirs('cmd/arena', exist_ok=True)
with open('cmd/arena/main.go', 'w', encoding='utf-8') as f:
    f.write('// Command arena: матч двух ботов с SPRT, сохранением партий и дампом проигранных позиций.\n//\n//\tgo run ./cmd/arena -a v2 -b v1 -depth 3 -pairs 300 -sprt 0,30 -games games.txt -losses losses.txt\n//\tgo run ./cmd/arena -a "v2:Walls=200" -b v2 -ms 30\npackage main\n\nimport (\n\t"flag"\n\t"fmt"\n\t"os"\n\t"runtime"\n\t"strconv"\n\t"strings"\n\n\t"quoridor/match"\n)\n\nfunc main() {\n\ta := flag.String("a", "v2", "бот A: walker | v1 | v2[:Name=Value,...]")\n\tb := flag.String("b", "v1", "бот B")\n\tdepth := flag.Int("depth", 3, "фиксированная глубина (0 = по времени)")\n\tms := flag.Int("ms", 0, "время на ход, мс (вместо глубины)")\n\tpairs := flag.Int("pairs", 200, "максимум пар партий")\n\tbatch := flag.Int("batch", 20, "размер пачки для проверки SPRT")\n\topening := flag.Int("opening", 4, "случайных полуходов дебюта")\n\tmaxPlies := flag.Int("maxplies", 300, "лимит полуходов (ничья)")\n\tsprt := flag.String("sprt", "0,30", "elo0,elo1 (alpha=beta=0.05); пусто = без SPRT, играть все пары")\n\tseed := flag.Int64("seed", 1, "seed дебютов")\n\tworkers := flag.Int("workers", runtime.GOMAXPROCS(0), "параллельных партий")\n\tgames := flag.String("games", "", "файл для сохранения партий")\n\tlosses := flag.String("losses", "", "файл для дампа проигранных позиций")\n\tlossWho := flag.String("losses-who", "A", "чьи поражения сохранять: A или B")\n\tlossN := flag.Int("losses-last", 4, "сколько последних позиций проигравшего сохранять в партии")\n\tflag.Parse()\n\n\tif *depth == 0 && *ms == 0 {\n\t\tfmt.Fprintln(os.Stderr, "нужен -depth или -ms")\n\t\tos.Exit(2)\n\t}\n\td := *depth\n\tif *ms > 0 {\n\t\td = 0\n\t}\n\tfa, err := match.BotFactory(*a, d, *ms, 20)\n\tif err != nil {\n\t\tfmt.Fprintln(os.Stderr, err)\n\t\tos.Exit(2)\n\t}\n\tfb, err := match.BotFactory(*b, d, *ms, 20)\n\tif err != nil {\n\t\tfmt.Fprintln(os.Stderr, err)\n\t\tos.Exit(2)\n\t}\n\n\tvar res match.SPRTResult\n\tif *sprt == "" {\n\t\tsc := match.Run(fa, fb, *pairs, *opening, *maxPlies, *workers, *seed)\n\t\tres = match.SPRTResult{Score: sc, Verdict: "n/a"}\n\t} else {\n\t\tparts := strings.Split(*sprt, ",")\n\t\te0, err0 := strconv.ParseFloat(parts[0], 64)\n\t\tvar e1 float64\n\t\tvar err1 error = fmt.Errorf("need elo0,elo1")\n\t\tif len(parts) == 2 {\n\t\t\te1, err1 = strconv.ParseFloat(parts[1], 64)\n\t\t}\n\t\tif err0 != nil || err1 != nil {\n\t\t\tfmt.Fprintln(os.Stderr, "bad -sprt, want elo0,elo1")\n\t\t\tos.Exit(2)\n\t\t}\n\t\tres = match.RunSPRT(fa, fb, match.SPRT{Elo0: e0, Elo1: e1, Alpha: 0.05, Beta: 0.05}, *pairs, *batch, *opening, *maxPlies, *workers, *seed)\n\t}\n\tfmt.Printf("%s vs %s: %s (%v)\\n", *a, *b, res, res.Score.Elapsed.Round(1e8))\n\n\t// RunSPRT не хранит партии (экономия памяти); для сохранения играем отдельно, если файл запрошен.\n\tif *games != "" || *losses != "" {\n\t\tsc := res.Score\n\t\tif len(sc.Games) == 0 {\n\t\t\tsc = match.Run(fa, fb, *pairs, *opening, *maxPlies, *workers, *seed)\n\t\t}\n\t\tif *games != "" {\n\t\t\tf, err := os.Create(*games)\n\t\t\tif err == nil {\n\t\t\t\terr = match.WriteGames(f, sc.Games)\n\t\t\t\tf.Close()\n\t\t\t}\n\t\t\treport(*games, len(sc.Games), err)\n\t\t}\n\t\tif *losses != "" {\n\t\t\twho := 0\n\t\t\tif strings.EqualFold(*lossWho, "B") {\n\t\t\t\twho = 1\n\t\t\t}\n\t\t\tps := match.LossPositions(sc.Games, who, *lossN)\n\t\t\tf, err := os.Create(*losses)\n\t\t\tif err == nil {\n\t\t\t\terr = match.WritePositions(f, ps)\n\t\t\t\tf.Close()\n\t\t\t}\n\t\t\treport(*losses, len(ps), err)\n\t\t}\n\t}\n}\n\nfunc report(name string, n int, err error) {\n\tif err != nil {\n\t\tfmt.Fprintln(os.Stderr, name+":", err)\n\t\treturn\n\t}\n\tfmt.Printf("saved %d records to %s\\n", n, name)\n}\n\n')

os.makedirs('cmd/server', exist_ok=True)
with open('cmd/server/main.go', 'w', encoding='utf-8') as f:
    f.write('// Command server: веб-интерфейс, в котором можно смотреть, как два бота играют друг с другом.\n//\n//\tgo run ./cmd/server            # затем открыть http://localhost:8080\n//\tgo run ./cmd/server -addr :9000\npackage main\n\nimport (\n\t"embed"\n\t"encoding/json"\n\t"flag"\n\t"fmt"\n\t"io/fs"\n\t"log"\n\t"net/http"\n\t"sync"\n\t"time"\n\n\t"quoridor/board"\n\t"quoridor/match"\n\t"quoridor/movegen"\n\t"quoridor/path"\n)\n\n//go:embed static\nvar staticFS embed.FS\n\n// ---- состояние партии ----\n\ntype game struct {\n\tmu       sync.Mutex\n\tb        board.Board\n\tbots     [2]match.Bot\n\tnames    [2]string\n\tmoves    []board.Move\n\ttexts    []string\n\tinfo     []moveInfo\n\twinner   int\n\tmaxPlies int\n\tlastMs   float64\n}\n\ntype moveInfo struct {\n\tPlayer int     `json:"player"`\n\tScore  *int    `json:"score,omitempty"` // оценка поиска с точки зрения ходившего\n\tDepth  int     `json:"depth,omitempty"`\n\tNodes  uint64  `json:"nodes,omitempty"`\n\tMs     float64 `json:"ms"`\n}\n\ntype newReq struct {\n\tP0       string `json:"p0"`\n\tP1       string `json:"p1"`\n\tDepth    int    `json:"depth"`\n\tMs       int    `json:"ms"`\n\tOpening  int    `json:"opening"` // случайных полуходов дебюта\n\tSeed     int64  `json:"seed"`\n\tMaxPlies int    `json:"maxPlies"`\n}\n\ntype server struct {\n\tmu sync.Mutex\n\tg  *game\n}\n\nfunc (s *server) newGame(r newReq) (*game, error) {\n\tif r.P0 == "" {\n\t\tr.P0 = "v2"\n\t}\n\tif r.P1 == "" {\n\t\tr.P1 = "v1"\n\t}\n\tif r.Depth == 0 && r.Ms == 0 {\n\t\tr.Depth = 3\n\t}\n\tif r.Depth < 0 || r.Depth > 12 || r.Ms < 0 || r.Ms > 60000 {\n\t\treturn nil, fmt.Errorf("depth must be 0..12 and ms 0..60000")\n\t}\n\tif r.MaxPlies <= 0 || r.MaxPlies > 1000 {\n\t\tr.MaxPlies = 300\n\t}\n\tif r.Opening < 0 || r.Opening > 20 {\n\t\treturn nil, fmt.Errorf("opening must be 0..20")\n\t}\n\tdepth := r.Depth\n\tif r.Ms > 0 {\n\t\tdepth = 0\n\t}\n\tg := &game{b: board.New(), winner: -1, maxPlies: r.MaxPlies, names: [2]string{r.P0, r.P1}}\n\tfor i, spec := range []string{r.P0, r.P1} {\n\t\tf, err := match.BotFactory(spec, depth, r.Ms, 20)\n\t\tif err != nil {\n\t\t\treturn nil, err\n\t\t}\n\t\tg.bots[i] = f()\n\t}\n\tif r.Opening > 0 {\n\t\tseed := r.Seed\n\t\tif seed == 0 {\n\t\t\tseed = time.Now().UnixNano()\n\t\t}\n\t\tg.playOpening(seed, r.Opening)\n\t}\n\treturn g, nil\n}\n\nfunc (g *game) playOpening(seed int64, n int) {\n\trec := match.RandomOpening(randSrc(seed), n)\n\tfor _, m := range rec {\n\t\tg.apply(m, moveInfo{Player: int(g.b.Turn)}, "дебют")\n\t}\n}\n\nfunc (g *game) apply(m board.Move, in moveInfo, tag string) {\n\tp := int(g.b.Turn)\n\tvar txt string\n\tswitch m.Kind() {\n\tcase board.KindPawn:\n\t\ttxt = fmt.Sprintf("пешка → (%d,%d)", m.Idx()/9, m.Idx()%9)\n\tcase board.KindH:\n\t\ttxt = fmt.Sprintf("гориз. стена (%d,%d)", m.Idx()/8, m.Idx()%8)\n\tdefault:\n\t\ttxt = fmt.Sprintf("вертик. стена (%d,%d)", m.Idx()/8, m.Idx()%8)\n\t}\n\tif tag != "" {\n\t\ttxt += " [" + tag + "]"\n\t}\n\tg.b = g.b.Apply(m)\n\tg.moves = append(g.moves, m)\n\tg.texts = append(g.texts, txt)\n\tin.Player = p\n\tg.info = append(g.info, in)\n\tif w := g.b.Winner(); w >= 0 {\n\t\tg.winner = w\n\t}\n}\n\n// step: один ход бота, чья очередь. Возвращает false, если партия уже окончена.\nfunc (g *game) step() bool {\n\tif g.winner >= 0 || len(g.moves) >= g.maxPlies {\n\t\treturn false\n\t}\n\tp := int(g.b.Turn)\n\tt0 := time.Now()\n\tm := g.bots[p].Choose(g.b)\n\tms := float64(time.Since(t0).Microseconds()) / 1000\n\tin := moveInfo{Ms: ms}\n\tif sb, ok := g.bots[p].(*match.Searcher); ok {\n\t\tsc := sb.Last.Score\n\t\tin.Score, in.Depth, in.Nodes = &sc, sb.Last.Depth, sb.Last.Nodes\n\t}\n\tg.apply(m, in, "")\n\treturn true\n}\n\n// ---- JSON ----\n\ntype stateJSON struct {\n\tNames    [2]string  `json:"names"`\n\tHW       [][2]int   `json:"hw"`\n\tVW       [][2]int   `json:"vw"`\n\tPawns    [2][2]int  `json:"pawns"`\n\tWalls    [2]int     `json:"walls"`\n\tTurn     int        `json:"turn"`\n\tWinner   int        `json:"winner"`\n\tPlies    int        `json:"plies"`\n\tDraw     bool       `json:"draw"`\n\tDist     [2]int     `json:"dist"`\n\tLast     *lastJSON  `json:"last"`\n\tHistory  []histJSON `json:"history"`\n\tMaxPlies int        `json:"maxPlies"`\n}\n\ntype lastJSON struct {\n\tKind   string `json:"kind"` // pawn | h | v\n\tR, C   int\n\tPlayer int `json:"player"`\n}\n\ntype histJSON struct {\n\tText string `json:"text"`\n\tmoveInfo\n}\n\nfunc (g *game) state() stateJSON {\n\ts := stateJSON{Names: g.names, Turn: int(g.b.Turn), Winner: g.winner, Plies: len(g.moves), MaxPlies: g.maxPlies,\n\t\tWalls: [2]int{int(g.b.Walls[0]), int(g.b.Walls[1])}, HW: [][2]int{}, VW: [][2]int{}}\n\tfor i := 0; i < 64; i++ {\n\t\tif g.b.HW>>uint(i)&1 != 0 {\n\t\t\ts.HW = append(s.HW, [2]int{i / 8, i % 8})\n\t\t}\n\t\tif g.b.VW>>uint(i)&1 != 0 {\n\t\t\ts.VW = append(s.VW, [2]int{i / 8, i % 8})\n\t\t}\n\t}\n\tfor p := 0; p < 2; p++ {\n\t\ts.Pawns[p] = [2]int{int(g.b.P[p]) / 9, int(g.b.P[p]) % 9}\n\t\ts.Dist[p] = path.Len(&g.b, p)\n\t}\n\ts.Draw = g.winner < 0 && len(g.moves) >= g.maxPlies\n\tif n := len(g.moves); n > 0 {\n\t\tm := g.moves[n-1]\n\t\tl := &lastJSON{Player: g.info[n-1].Player}\n\t\tswitch m.Kind() {\n\t\tcase board.KindPawn:\n\t\t\tl.Kind, l.R, l.C = "pawn", m.Idx()/9, m.Idx()%9\n\t\tcase board.KindH:\n\t\t\tl.Kind, l.R, l.C = "h", m.Idx()/8, m.Idx()%8\n\t\tdefault:\n\t\t\tl.Kind, l.R, l.C = "v", m.Idx()/8, m.Idx()%8\n\t\t}\n\t\ts.Last = l\n\t}\n\ts.History = []histJSON{}\n\tfor i := range g.moves {\n\t\ts.History = append(s.History, histJSON{Text: g.texts[i], moveInfo: g.info[i]})\n\t}\n\treturn s\n}\n\n// ---- HTTP ----\n\nfunc (s *server) handler() http.Handler {\n\tmux := http.NewServeMux()\n\tsub, _ := fs.Sub(staticFS, "static")\n\tmux.Handle("/", http.FileServer(http.FS(sub)))\n\n\twrite := func(w http.ResponseWriter, code int, v any) {\n\t\tw.Header().Set("Content-Type", "application/json")\n\t\tw.WriteHeader(code)\n\t\tjson.NewEncoder(w).Encode(v)\n\t}\n\tfail := func(w http.ResponseWriter, code int, err error) {\n\t\twrite(w, code, map[string]string{"error": err.Error()})\n\t}\n\tcur := func() *game {\n\t\ts.mu.Lock()\n\t\tdefer s.mu.Unlock()\n\t\tif s.g == nil {\n\t\t\ts.g, _ = s.newGame(newReq{})\n\t\t}\n\t\treturn s.g\n\t}\n\n\tmux.HandleFunc("/api/new", func(w http.ResponseWriter, r *http.Request) {\n\t\tif r.Method != http.MethodPost {\n\t\t\tfail(w, http.StatusMethodNotAllowed, fmt.Errorf("POST only"))\n\t\t\treturn\n\t\t}\n\t\tvar req newReq\n\t\tr.Body = http.MaxBytesReader(w, r.Body, 1<<16)\n\t\tif err := json.NewDecoder(r.Body).Decode(&req); err != nil {\n\t\t\tfail(w, http.StatusBadRequest, err)\n\t\t\treturn\n\t\t}\n\t\tg, err := s.newGame(req)\n\t\tif err != nil {\n\t\t\tfail(w, http.StatusBadRequest, err)\n\t\t\treturn\n\t\t}\n\t\ts.mu.Lock()\n\t\ts.g = g\n\t\ts.mu.Unlock()\n\t\twrite(w, 200, g.state())\n\t})\n\tmux.HandleFunc("/api/state", func(w http.ResponseWriter, r *http.Request) {\n\t\tg := cur()\n\t\tg.mu.Lock()\n\t\tdefer g.mu.Unlock()\n\t\twrite(w, 200, g.state())\n\t})\n\tmux.HandleFunc("/api/step", func(w http.ResponseWriter, r *http.Request) {\n\t\tif r.Method != http.MethodPost {\n\t\t\tfail(w, http.StatusMethodNotAllowed, fmt.Errorf("POST only"))\n\t\t\treturn\n\t\t}\n\t\tg := cur()\n\t\tg.mu.Lock()\n\t\tdefer g.mu.Unlock()\n\t\tg.step()\n\t\twrite(w, 200, g.state())\n\t})\n\treturn mux\n}\n\nfunc main() {\n\taddr := flag.String("addr", "127.0.0.1:8080", "адрес сервера (для доступа из сети: :8080)")\n\tflag.Parse()\n\ts := &server{}\n\tg, err := s.newGame(newReq{})\n\tif err != nil {\n\t\tlog.Fatal(err)\n\t}\n\ts.g = g\n\t_ = movegen.Gen // (держим зависимость явной: сервер опирается на те же правила, что и боты)\n\tlog.Printf("Quoridor: http://%s", *addr)\n\tlog.Fatal(http.ListenAndServe(*addr, s.handler()))\n}\n\n')

os.makedirs('cmd/server', exist_ok=True)
with open('cmd/server/rand.go', 'w', encoding='utf-8') as f:
    f.write('package main\n\nimport "math/rand"\n\nfunc randSrc(seed int64) *rand.Rand { return rand.New(rand.NewSource(seed)) }\n\n')

os.makedirs('cmd/server', exist_ok=True)
with open('cmd/server/main_test.go', 'w', encoding='utf-8') as f:
    f.write('package main\n\nimport (\n\t"bytes"\n\t"encoding/json"\n\t"net/http"\n\t"net/http/httptest"\n\t"testing"\n)\n\nfunc post(t *testing.T, h http.Handler, path string, body any) (int, stateJSON, map[string]string) {\n\tt.Helper()\n\tvar buf bytes.Buffer\n\tjson.NewEncoder(&buf).Encode(body)\n\treq := httptest.NewRequest(http.MethodPost, path, &buf)\n\tw := httptest.NewRecorder()\n\th.ServeHTTP(w, req)\n\tvar s stateJSON\n\tvar e map[string]string\n\tif w.Code == 200 {\n\t\tjson.Unmarshal(w.Body.Bytes(), &s)\n\t} else {\n\t\tjson.Unmarshal(w.Body.Bytes(), &e)\n\t}\n\treturn w.Code, s, e\n}\n\nfunc TestPlayFullGameOverHTTP(t *testing.T) {\n\tsrv := &server{}\n\th := srv.handler()\n\tcode, s, _ := post(t, h, "/api/new", newReq{P0: "v2", P1: "walker", Depth: 2, Opening: 2, Seed: 1})\n\tif code != 200 || s.Plies != 2 {\n\t\tt.Fatalf("new: code %d plies %d", code, s.Plies)\n\t}\n\tfor i := 0; i < 400 && s.Winner < 0 && !s.Draw; i++ {\n\t\tcode, s, _ = post(t, h, "/api/step", struct{}{})\n\t\tif code != 200 {\n\t\t\tt.Fatalf("step code %d", code)\n\t\t}\n\t}\n\tif s.Winner < 0 && !s.Draw {\n\t\tt.Fatal("game did not finish")\n\t}\n\tif len(s.History) != s.Plies || s.Last == nil {\n\t\tt.Fatalf("history %d plies %d last %v", len(s.History), s.Plies, s.Last)\n\t}\n\t// после конца партии шаг ничего не меняет\n\t_, s2, _ := post(t, h, "/api/step", struct{}{})\n\tif s2.Plies != s.Plies {\n\t\tt.Fatal("step after the end must be a no-op")\n\t}\n}\n\nfunc TestBadRequests(t *testing.T) {\n\th := (&server{}).handler()\n\tfor _, r := range []newReq{{P0: "nope"}, {P0: "v2:Flex=x"}, {Depth: 99}, {Opening: 99}} {\n\t\tif code, _, e := post(t, h, "/api/new", r); code != 400 || e["error"] == "" {\n\t\t\tt.Errorf("%+v: code %d err %v", r, code, e)\n\t\t}\n\t}\n\treq := httptest.NewRequest(http.MethodGet, "/api/step", nil)\n\tw := httptest.NewRecorder()\n\th.ServeHTTP(w, req)\n\tif w.Code != http.StatusMethodNotAllowed {\n\t\tt.Errorf("GET /api/step: %d", w.Code)\n\t}\n}\n\nfunc TestIndexServed(t *testing.T) {\n\th := (&server{}).handler()\n\tw := httptest.NewRecorder()\n\th.ServeHTTP(w, httptest.NewRequest(http.MethodGet, "/", nil))\n\tif w.Code != 200 || !bytes.Contains(w.Body.Bytes(), []byte("Quoridor Arena")) {\n\t\tt.Fatalf("index: %d", w.Code)\n\t}\n}\n\n')

os.makedirs('cmd/tune', exist_ok=True)
with open('cmd/tune/main.go', 'w', encoding='utf-8') as f:
    f.write('// Command tune: настройка весов eval.V2 методом SPSA (self-play theta+ против theta-).\n//\n//\tgo run ./cmd/tune -iters 150 -pairs 20 -depth 3\n//\n// Каждые -report итераций печатает текущий вектор как готовую строку eval.Params.\npackage main\n\nimport (\n\t"flag"\n\t"fmt"\n\t"runtime"\n\n\t"quoridor/tune"\n)\n\nfunc main() {\n\titers := flag.Int("iters", 150, "число итераций SPSA")\n\tpairs := flag.Int("pairs", 20, "пар партий на итерацию (партий = 2*pairs)")\n\tdepth := flag.Int("depth", 3, "глубина поиска ботов")\n\topening := flag.Int("opening", 4, "случайных полуходов дебюта")\n\tlr := flag.Float64("lr", 1.0, "начальный шаг в долях размаха возмущения")\n\tseed := flag.Int64("seed", 1, "seed")\n\tworkers := flag.Int("workers", runtime.GOMAXPROCS(0), "параллельных партий")\n\treport := flag.Int("report", 10, "как часто печатать вектор")\n\tflag.Parse()\n\n\tspecs := tune.DefaultSpecs()\n\ts := tune.New(specs, *lr, *iters, *seed)\n\tit := int64(0)\n\tscore := tune.MatchScore(specs, *depth, *pairs, *opening, *workers, func() int64 { it++; return *seed*100000 + it*997 })\n\tfor k := 0; k < *iters; k++ {\n\t\tsc := s.Step(score)\n\t\tif (k+1)%*report == 0 || k+1 == *iters {\n\t\t\tfmt.Printf("iter %3d (last match %.2f): %s\\n", k+1, sc, tune.ToParams(specs, s.Theta))\n\t\t}\n\t}\n\tfmt.Println("final:", tune.ToParams(specs, s.Theta))\n}\n\n')



In [ ]:
import modal
import os

# Инициализация приложения Modal
app = modal.App("quoridor-spsa-tune-notebook")

# Создаем легковесный контейнер Debian и устанавливаем Go 1.22
go_image = (
    modal.Image.debian_slim()
    .apt_install(
        "wget",
        "tar"
    )
    .run_commands(
        "wget -q https://go.dev/dl/go1.22.4.linux-amd64.tar.gz",
        "tar -C /usr/local -xzf go1.22.4.linux-amd64.tar.gz",
        "rm go1.22.4.linux-amd64.tar.gz",
    )
    .env({"PATH": "/usr/local/go/bin:" + os.environ.get("PATH", "")})
)

# Выделяем мощный инстанс на 16 CPU и 8 ГБ RAM
@app.function(image=go_image, cpu=16.0, memory=8192, timeout=7200)
def run_tune_remote(iters: int = 1000, pairs: int = 30, depth: int = 3, workers: int = 16):
    import subprocess
    print("=== Проверка тестов Go ===")
    subprocess.run(["go", "test", "./..."])
    
    print(f"\n=== Запуск SPSA тюнинга на {workers} ядрах в облаке Modal ===")
    cmd = f"go run ./cmd/tune -iters {iters} -pairs {pairs} -depth {depth} -workers {workers}"
    subprocess.run(cmd, shell=True)


In [ ]:
# Запуск вычислений на мощном облачном сервере Modal
with app.run():
    run_tune_remote.remote(iters=1000, pairs=30, depth=3, workers=16)
